# CitraSatelit — CBIR dan Berbagi Rahasia dengan XOR

Gunakan EuroSAT untuk menguji histogram, pencarian citra, Bayes, dan rekonstruksi XOR.
Enam percobaan di bagian akhir memakai protokol serta modul yang sama dengan aplikasi web.

## 1. Setup dan pilihan data

Pilih **Jalankan semua** setelah menyediakan arsip. Untuk Colab, unggah **eurosat_subset_v1.zip** ke **/content**, atau isi
**COLAB_SUBSET_URL** dengan URL rilis Hugging Face dan **COLAB_SUBSET_MIRROR** dengan cermin GitHub Releases.
Simpan **.sha256** di samping arsip rilis, atau isi **COLAB_SUBSET_SHA256** dengan hash yang sudah diketahui.
Mode **colab_instant** memakai 1.600 citra latih dan 400 citra uji.
Mode **full_local** membaca direktori EuroSAT dan indeks yang telah dibangun melalui konfigurasi proyek.
**REPO_URL** opsional untuk mengkloning repositori. Sumber yang tertanam digunakan jika URL kosong.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path
from tempfile import TemporaryDirectory

DATASET_MODE = os.getenv("DATASET_MODE", "colab_instant")
REPO_URL = os.getenv("CITRASATELIT_REPO_URL", "")
COLAB_SUBSET_URL = os.getenv("COLAB_SUBSET_URL", "")
COLAB_SUBSET_MIRROR = os.getenv("COLAB_SUBSET_MIRROR", "")
COLAB_SUBSET_SHA256 = os.getenv("COLAB_SUBSET_SHA256", "")
COLAB_ARCHIVE = os.getenv("COLAB_ARCHIVE", "")
in_colab = Path("/content").is_dir()
workspace = Path(os.getenv("CITRASATELIT_PROJECT", str(Path.cwd()))).resolve()
cache = Path(os.getenv("COLAB_CACHE_DIR", str(Path("/content/data") if in_colab else workspace / ".colab-cache")))
cache.mkdir(parents=True, exist_ok=True)

if in_colab:
    import importlib.util
    dependencies = {"numpy": "numpy", "PIL": "pillow", "tifffile": "tifffile",
                    "matplotlib": "matplotlib", "dotenv": "python-dotenv"}
    missing = [package for module, package in dependencies.items() if importlib.util.find_spec(module) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
embedded_sources = {}

## 2. Sumber modul
Sel berikut dihasilkan dari **src/core**. Sunting modul proyek lalu jalankan **python -m tools.build_notebook** untuk memperbarui notebook.

In [ ]:
embedded_sources['src/core/boolean_lab.py'] = (
    'from __future__ import annotations\n'
    '\n'
    'from dataclasses import dataclass\n'
    'from itertools import combinations, product\n'
    'from numbers import Integral\n'
    '\n'
    '\n'
    'class ExpressionError(ValueError):\n'
    '    """An invalid Boolean expression or assignment."""\n'
    '\n'
    '\n'
    '@dataclass(frozen=True)\n'
    'class _Node:\n'
    '    op: str\n'
    '    children: tuple[_Node, ...] = ()\n'
    '    label: str = ""\n'
    '\n'
    '\n'
    '_OPERATORS = {"+": "or", "^": "xor", ".": "and"}\n'
    '_PRECEDENCE = {"or": 1, "xor": 2, "and": 3, "not": 4, "input": 5}\n'
    '_SYMBOLS = {"or": "+", "xor": "^", "and": ""}\n'
    '\n'
    '\n'
    'class _Parser:\n'
    '    def __init__(self, expression: str):\n'
    '        aliases = str.maketrans({"*": ".", "~": "!", "\\u2295": "^"})\n'
    '        self.source = "".join(expression.split()).translate(aliases)\n'
    '        self.index = 0\n'
    '\n'
    '    def peek(self) -> str:\n'
    '        return self.source[self.index:self.index + 1]\n'
    '\n'
    '    def expression(self, minimum: int = 1) -> _Node:\n'
    '        left = self.unary()\n'
    '        while True:\n'
    '            token = self.peek()\n'
    '            op = _OPERATORS.get(token)\n'
    '            implicit = token.isalpha() or token in ("(", "!", "0", "1")\n'
    '            if implicit:\n'
    '                op = "and"\n'
    '            if op is None or _PRECEDENCE[op] < minimum:\n'
    '                return left\n'
    '            if not implicit:\n'
    '                self.index += 1\n'
    '            right = self.expression(_PRECEDENCE[op] + 1)\n'
    '            left = _Node(op, (left, right))\n'
    '\n'
    '    def unary(self) -> _Node:\n'
    '        token = self.peek()\n'
    '        self.index += 1\n'
    '        if token == "!":\n'
    '            return _Node("not", (self.unary(),))\n'
    '        if token == "(":\n'
    '            node = self.expression()\n'
    '            if self.peek() != ")":\n'
    '                raise ExpressionError("expected closing parenthesis")\n'
    '            self.index += 1\n'
    '        elif token.isalpha() or token in ("0", "1"):\n'
    '            node = _Node("input", label=token)\n'
    '        else:\n'
    '            raise ExpressionError(f"expected an operand, got {token or \'end of expression\'!r}")\n'
    '\n'
    '        while self.peek() == "\'":\n'
    '            self.index += 1\n'
    '            node = _Node("not", (node,))\n'
    '        return node\n'
    '\n'
    '\n'
    'def _variables(node: _Node) -> set[str]:\n'
    '    variables = set()\n'
    '    pending = [node]\n'
    '    while pending:\n'
    '        current = pending.pop()\n'
    '        if current.op == "input" and current.label.isalpha():\n'
    '            variables.add(current.label)\n'
    '        pending.extend(current.children)\n'
    '    return variables\n'
    '\n'
    '\n'
    'def _ordered_variables(found: set[str], variables=None) -> tuple[str, ...]:\n'
    '    names = tuple(sorted(found)) if variables is None else tuple(variables)\n'
    '    if len(names) > 6:\n'
    '        raise ExpressionError("at most six variables are supported")\n'
    '    if any(not isinstance(name, str) or len(name) != 1 or not name.isalpha() for name in names):\n'
    '        raise ExpressionError("variables must be single letters")\n'
    '    if len(set(names)) != len(names) or not found.issubset(names):\n'
    '        raise ExpressionError("variables must be unique and contain all expression variables")\n'
    '    return names\n'
    '\n'
    '\n'
    'def parse_expr(expression: str) -> _Node:\n'
    '    """Parse one expression with the precedence NOT, AND, XOR, OR."""\n'
    '    if not isinstance(expression, str):\n'
    '        raise ExpressionError("expression must be a string")\n'
    '    parser = _Parser(expression)\n'
    '    node = parser.expression()\n'
    '    if parser.peek():\n'
    '        raise ExpressionError(f"unexpected token {parser.peek()!r}")\n'
    '    _ordered_variables(_variables(node))\n'
    '    return node\n'
    '\n'
    '\n'
    'def _tree(expression) -> _Node:\n'
    '    return expression if isinstance(expression, _Node) else parse_expr(expression)\n'
    '\n'
    '\n'
    'def _evaluate(node: _Node, values) -> int:\n'
    '    if node.op == "input":\n'
    '        if node.label in ("0", "1"):\n'
    '            return int(node.label)\n'
    '        if node.label not in values or values[node.label] not in (0, 1):\n'
    '            raise ExpressionError(f"expected a bit for variable {node.label!r}")\n'
    '        return int(values[node.label])\n'
    '    if node.op == "not":\n'
    '        return 1 - _evaluate(node.children[0], values)\n'
    '\n'
    '    left, right = (_evaluate(child, values) for child in node.children)\n'
    '    if node.op == "and":\n'
    '        return left & right\n'
    '    if node.op == "xor":\n'
    '        return left ^ right\n'
    '    return left | right\n'
    '\n'
    '\n'
    'def eval_expr(expression, values) -> int:\n'
    '    """Evaluate an expression string or a parsed expression for one assignment."""\n'
    '    return _evaluate(_tree(expression), values)\n'
    '\n'
    '\n'
    'def _format(node: _Node, minimum: int = 0) -> str:\n'
    '    precedence = _PRECEDENCE[node.op]\n'
    '    if node.op == "input":\n'
    '        return node.label\n'
    '    if node.op == "not":\n'
    '        text = _format(node.children[0], precedence) + "\'"\n'
    '    else:\n'
    '        text = _SYMBOLS[node.op].join(_format(child, precedence) for child in node.children)\n'
    '    return f"({text})" if precedence < minimum else text\n'
    '\n'
    '\n'
    'def truth_table(expression, variables=None) -> dict:\n'
    '    """Return rows in binary order with the first variable as the most significant bit."""\n'
    '    node = _tree(expression)\n'
    '    names = _ordered_variables(_variables(node), variables)\n'
    '    rows = [\n'
    '        {"inputs": list(bits), "output": _evaluate(node, dict(zip(names, bits)))}\n'
    '        for bits in product((0, 1), repeat=len(names))\n'
    '    ]\n'
    '    return {"normalized": _format(node), "variables": list(names), "rows": rows}\n'
    '\n'
    '\n'
    'def minterms(expression, variables=None) -> list[int]:\n'
    '    return [index for index, row in enumerate(truth_table(expression, variables)["rows"]) if row["output"]]\n'
    '\n'
    '\n'
    'def _product_term(names, bits) -> str:\n'
    '    return "".join(name if bit else name + "\'" for name, bit in zip(names, bits) if bit is not None) or "1"\n'
    '\n'
    '\n'
    'def sop_canonical(expression, variables=None) -> str:\n'
    '    table = truth_table(expression, variables)\n'
    '    terms = [\n'
    '        _product_term(table["variables"], row["inputs"])\n'
    '        for row in table["rows"]\n'
    '        if row["output"]\n'
    '    ]\n'
    '    return "+".join(terms) or "0"\n'
    '\n'
    '\n'
    'def equivalent(lhs, rhs) -> bool:\n'
    '    left, right = _tree(lhs), _tree(rhs)\n'
    '    names = _ordered_variables(_variables(left) | _variables(right))\n'
    '    for bits in product((0, 1), repeat=len(names)):\n'
    '        values = dict(zip(names, bits))\n'
    '        if _evaluate(left, values) != _evaluate(right, values):\n'
    '            return False\n'
    '    return True\n'
    '\n'
    '\n'
    'def _dual(node: _Node) -> _Node:\n'
    '    if node.op == "input":\n'
    '        return _Node("input", label={"0": "1", "1": "0"}.get(node.label, node.label))\n'
    '\n'
    '    children = tuple(_dual(child) for child in node.children)\n'
    '    if node.op == "xor":\n'
    '        # Complementing both inputs leaves XOR unchanged, so its dual is XNOR.\n'
    '        return _Node("not", (_Node("xor", children),))\n'
    '    return _Node({"and": "or", "or": "and", "not": "not"}[node.op], children)\n'
    '\n'
    '\n'
    'def dual_expr(expression) -> str:\n'
    '    return _format(_dual(_tree(expression)))\n'
    '\n'
    '\n'
    'def to_circuit(expression) -> dict:\n'
    '    """Build the Circuit DAG, using labelled input nodes for variables and constants."""\n'
    '    nodes = []\n'
    '    edges = []\n'
    '    inputs = {}\n'
    '\n'
    '    def visit(node: _Node) -> str:\n'
    '        if node.op == "input" and node.label in inputs:\n'
    '            return inputs[node.label]\n'
    '\n'
    '        sources = [visit(child) for child in node.children]\n'
    '        node_id = f"n{len(nodes)}"\n'
    '        nodes.append({"id": node_id, "type": node.op, "label": node.label or node.op.upper()})\n'
    '        edges.extend({"from": source, "to": node_id} for source in sources)\n'
    '        if node.op == "input":\n'
    '            inputs[node.label] = node_id\n'
    '        return node_id\n'
    '\n'
    '    output_id = visit(_tree(expression))\n'
    '    return {"nodes": nodes, "edges": edges, "output_id": output_id}\n'
    '\n'
    '\n'
    'def _prime_implicants(terms: set[int]) -> set[tuple[int, int]]:\n'
    '    # A pair contains fixed bits and a mask of positions that can vary.\n'
    '    layer = {(term, 0) for term in terms}\n'
    '    primes = set()\n'
    '    while layer:\n'
    '        merged = set()\n'
    '        next_layer = set()\n'
    '        for first, second in combinations(layer, 2):\n'
    '            left, mask = first\n'
    '            right, other_mask = second\n'
    '            difference = left ^ right\n'
    '            if mask == other_mask and difference and difference.bit_count() == 1:\n'
    '                merged.update((first, second))\n'
    '                next_layer.add((left & ~difference, mask | difference))\n'
    '        primes.update(layer - merged)\n'
    '        layer = next_layer\n'
    '    return primes\n'
    '\n'
    '\n'
    'def _implicant_group(implicant, names, terms, coordinates) -> dict:\n'
    '    fixed, mask = implicant\n'
    '    bit_values = [\n'
    '        None if mask & (1 << shift) else (fixed >> shift) & 1\n'
    '        for shift in range(len(names) - 1, -1, -1)\n'
    '    ]\n'
    '    cells = sorted(term for term in terms if term & ~mask == fixed)\n'
    '    positions = [coordinates[cell] for cell in cells]\n'
    '    wraps = any(max(axis) - min(axis) + 1 > len(set(axis)) for axis in zip(*positions))\n'
    '    return {\n'
    '        "cells": cells,\n'
    '        "cells_rc": positions,\n'
    '        "term": _product_term(names, bit_values),\n'
    '        "size": len(cells),\n'
    '        "wraps": wraps,\n'
    '    }\n'
    '\n'
    '\n'
    'def _select_implicants(groups, terms) -> list[dict]:\n'
    '    cells = [set(group["cells"]) for group in groups]\n'
    '    selected = set()\n'
    '    for term in terms:\n'
    '        candidates = [index for index, covered in enumerate(cells) if term in covered]\n'
    '        if len(candidates) == 1:\n'
    '            selected.add(candidates[0])\n'
    '\n'
    '    uncovered = terms.difference(*(cells[index] for index in selected))\n'
    '    while uncovered:\n'
    '        # Prefer greater new coverage, then larger groups, then expression order.\n'
    '        index = min(\n'
    '            (index for index in range(len(groups)) if index not in selected),\n'
    '            key=lambda index: (-len(uncovered & cells[index]), -len(cells[index]), groups[index]["term"]),\n'
    '        )\n'
    '        selected.add(index)\n'
    '        uncovered -= cells[index]\n'
    '    return [groups[index] for index in sorted(selected)]\n'
    '\n'
    '\n'
    'def _gray_labels(width: int) -> list[str]:\n'
    '    return [f"{index ^ (index >> 1):0{width}b}" for index in range(1 << width)]\n'
    '\n'
    '\n'
    'def kmap(variables, minterms) -> dict:\n'
    '    """Arrange 2-4 variables in Gray order and simplify their on-set."""\n'
    '    if isinstance(variables, Integral):\n'
    '        if not 2 <= variables <= 4:\n'
    '            raise ExpressionError("K-Maps require two to four variables")\n'
    '        variables = "abcd"[:variables]\n'
    '    names = _ordered_variables(set(), variables)\n'
    '    if not 2 <= len(names) <= 4:\n'
    '        raise ExpressionError("K-Maps require two to four variables")\n'
    '\n'
    '    terms = set()\n'
    '    for term in minterms:\n'
    '        if not isinstance(term, Integral) or not 0 <= term < 1 << len(names):\n'
    '            raise ExpressionError("minterms must be within the K-Map")\n'
    '        terms.add(int(term))\n'
    '\n'
    '    row_width = len(names) // 2\n'
    '    rows = _gray_labels(row_width)\n'
    '    columns = _gray_labels(len(names) - row_width)\n'
    '    cells = [[int(row + column, 2) for column in columns] for row in rows]\n'
    '    coordinates = {\n'
    '        term: [row, col] for row, line in enumerate(cells) for col, term in enumerate(line)\n'
    '    }\n'
    '    groups = sorted(\n'
    '        (_implicant_group(prime, names, terms, coordinates) for prime in _prime_implicants(terms)),\n'
    '        key=lambda group: group["term"],\n'
    '    )\n'
    '    selected = _select_implicants(groups, terms)\n'
    '\n'
    '    return {\n'
    '        "variables": list(names),\n'
    '        "row_vars": list(names[:row_width]),\n'
    '        "col_vars": list(names[row_width:]),\n'
    '        "row_labels": rows,\n'
    '        "col_labels": columns,\n'
    '        "grid": [[int(term in terms) for term in line] for line in cells],\n'
    '        "cell_minterm": cells,\n'
    '        "prime_implicants": groups,\n'
    '        "selected": selected,\n'
    '        "simplified": " + ".join(group["term"] for group in selected) or "0",\n'
    '    }\n'
    '\n'
    '\n'
    'XOR_DEFINITION = "ab\'+a\'b"\n'
    '\n'
    '\n'
    'def _proof(*steps) -> list[dict[str, str]]:\n'
    '    return [{"expression": expression, "reason": reason} for expression, reason in steps]\n'
    '\n'
    '\n'
    'LAW_PROOFS = {\n'
    '    "a+a\'b=a+b": _proof(\n'
    '        ("a+a\'b", "Ekspresi awal."),\n'
    '        ("(a+a\')(a+b)", "Hukum distributif."),\n'
    '        ("1(a+b)", "Komplemen a+a\' bernilai 1."),\n'
    '        ("a+b", "Identitas perkalian."),\n'
    '    ),\n'
    '    "a(a\'+b)=ab": _proof(\n'
    '        ("a(a\'+b)", "Ekspresi awal."),\n'
    '        ("aa\'+ab", "Hukum distributif."),\n'
    '        ("0+ab", "Komplemen aa\' bernilai 0."),\n'
    '        ("ab", "Identitas penjumlahan."),\n'
    '    ),\n'
    '    "(a+b)\'=a\'b\'": _proof(\n'
    '        ("(a+b)\'", "Ekspresi awal."),\n'
    '        ("a\'b\'", "Komplemen unik. Hasil kali (a+b)a\'b\' bernilai 0, dan jumlah "\n'
    '         "(a+b)+a\'b\' = (a+b+a\')(a+b+b\') bernilai 1."),\n'
    '    ),\n'
    '    "(ab)\'=a\'+b\'": _proof(\n'
    '        ("(ab)\'", "Ekspresi awal."),\n'
    '        ("a\'+b\'", "Komplemen unik. Hasil kali ab(a\'+b\') bernilai 0, dan jumlah "\n'
    '         "ab+(a\'+b\') = (a+a\'+b\')(b+a\'+b\') bernilai 1."),\n'
    '    ),\n'
    '    "a+ab=a": _proof(\n'
    '        ("a+ab", "Ekspresi awal."),\n'
    '        ("a(1+b)", "Faktorkan a."),\n'
    '        ("a1", "Hukum dominasi."),\n'
    '        ("a", "Identitas perkalian."),\n'
    '    ),\n'
    '    "a(a+b)=a": _proof(\n'
    '        ("a(a+b)", "Ekspresi awal."),\n'
    '        ("aa+ab", "Hukum distributif."),\n'
    '        ("a+ab", "Idempotensi."),\n'
    '        ("a", "Penyerapan penjumlahan."),\n'
    '    ),\n'
    '    "a\\u2295b=(a+b)(ab)\'": _proof(\n'
    '        ("a\\u2295b", "Ekspresi awal."),\n'
    '        (XOR_DEFINITION, "Definisi XOR."),\n'
    '        ("aa\'+ab\'+ba\'+bb\'", "Tambahkan suku nol aa\' dan bb\', lalu gunakan komutativitas."),\n'
    '        ("(a+b)(a\'+b\')", "Faktorkan dengan hukum distributif."),\n'
    '        ("(a+b)(ab)\'", "Gunakan hukum De Morgan."),\n'
    '    ),\n'
    '    "a\\u2295b=ab\'+a\'b": _proof(\n'
    '        ("a\\u2295b", "Ekspresi awal."),\n'
    '        (XOR_DEFINITION, "Definisi XOR."),\n'
    '    ),\n'
    '}\n'
)

In [ ]:
embedded_sources['src/core/cbir.py'] = (
    'import numpy as np\n'
    '\n'
    'from src.core.color import grid_cells, rgb_to_gray, rgb_to_hsv\n'
    'from src.core.spectral import ndi_to_level, ndvi, ndwi, scale_to_8bit, synth_rgb\n'
    'from src.core.stats import coarsen_counts, counts_to_pmf, histogram_counts, mle_gaussian, moments_from_counts\n'
    '\n'
    '\n'
    '_PMF_METRICS = ("euclid", "chi2", "intersection")\n'
    '_MULTISPECTRAL_MODES = {"ms13", "ndvi_ndwi", "ms13_idx", "ms_mle"}\n'
    'MODE_INFO = {\n'
    '    mode: {\n'
    '        "kind": kind,\n'
    '        "metrics": metrics,\n'
    '        "needs_multispectral": mode in _MULTISPECTRAL_MODES,\n'
    '    }\n'
    '    for kind, modes, metrics in (\n'
    '        ("pmf", ("gray", "rgb", "hsv", "ms13", "ndvi_ndwi", "ms13_idx"), _PMF_METRICS),\n'
    '        ("moment", ("grid2", "grid4", "mle", "ms_mle"), ("euclid",)),\n'
    '    )\n'
    '    for mode in modes\n'
    '}\n'
    '\n'
    '\n'
    'def _band_image(bands) -> np.ndarray:\n'
    '    if bands is None:\n'
    '        raise ValueError("multispectral modes require bands")\n'
    '    image = np.asarray(bands)\n'
    '    if image.ndim != 3 or image.shape[-1] != 13 or image.size == 0:\n'
    '        raise ValueError("bands must be a nonempty (H, W, 13) image")\n'
    '    return image\n'
    '\n'
    '\n'
    'def _require_scale(scale):\n'
    '    if scale is None:\n'
    '        raise ValueError("8-bit band conversion requires a fitted scale")\n'
    '    return scale\n'
    '\n'
    '\n'
    'def _rgb_image(rgb, bands, scale) -> np.ndarray:\n'
    '    if rgb is None:\n'
    '        if bands is None:\n'
    '            raise ValueError("RGB modes require rgb or bands with a fitted scale")\n'
    '        return synth_rgb(_band_image(bands), _require_scale(scale))\n'
    '\n'
    '    image = np.asarray(rgb)\n'
    '    if image.ndim != 3 or image.shape[-1] != 3 or image.size == 0:\n'
    '        raise ValueError("rgb must be a nonempty (H, W, 3) image")\n'
    '    return image\n'
    '\n'
    '\n'
    'def _channel_counts(image) -> np.ndarray:\n'
    '    return np.stack([histogram_counts(channel, 256) for channel in np.moveaxis(image, -1, 0)])\n'
    '\n'
    '\n'
    'def extract_counts(mode, *, rgb=None, bands=None, scale=None) -> np.ndarray:\n'
    '    """Return full 256-level counts, one row per channel in multichannel modes."""\n'
    '    if mode not in MODE_INFO or MODE_INFO[mode]["kind"] != "pmf":\n'
    '        raise ValueError(f"not a PMF feature mode: {mode}")\n'
    '\n'
    '    if not MODE_INFO[mode]["needs_multispectral"]:\n'
    '        image = _rgb_image(rgb, bands, scale)\n'
    '        if mode == "gray":\n'
    '            return histogram_counts(rgb_to_gray(image), 256)\n'
    '        if mode == "hsv":\n'
    '            image = rgb_to_hsv(image)\n'
    '        return _channel_counts(image)\n'
    '\n'
    '    image = _band_image(bands)\n'
    '    if mode == "ms13":\n'
    '        return _channel_counts(scale_to_8bit(image, _require_scale(scale)))\n'
    '\n'
    '    indices = np.stack([\n'
    '        histogram_counts(ndi_to_level(ndvi(image)), 256),\n'
    '        histogram_counts(ndi_to_level(ndwi(image)), 256),\n'
    '    ])\n'
    '    if mode == "ndvi_ndwi":\n'
    '        return indices\n'
    '    band_counts = _channel_counts(scale_to_8bit(image, _require_scale(scale)))\n'
    '    return np.concatenate((band_counts, indices))\n'
    '\n'
    '\n'
    'def counts_to_feature(counts, bins) -> np.ndarray:\n'
    '    """Coarsen full counts and concatenate equally weighted channel PMFs."""\n'
    '    coarse = coarsen_counts(counts, bins)\n'
    '    if coarse.ndim == 1:\n'
    '        return counts_to_pmf(coarse)\n'
    '    if coarse.ndim != 2 or len(coarse) == 0:\n'
    '        raise ValueError("expected a histogram or a nonempty matrix of channel histograms")\n'
    '\n'
    '    pmfs = np.concatenate([counts_to_pmf(channel) for channel in coarse])\n'
    '    return pmfs / len(coarse)\n'
    '\n'
    '\n'
    'def extract_moment_feature(mode, *, rgb=None, bands=None, scale=None) -> np.ndarray:\n'
    '    """Return grid moments or channel-wise Gaussian MLE features."""\n'
    '    if mode not in MODE_INFO or MODE_INFO[mode]["kind"] != "moment":\n'
    '        raise ValueError(f"not a moment feature mode: {mode}")\n'
    '\n'
    '    if mode == "ms_mle":\n'
    '        image = scale_to_8bit(_band_image(bands), _require_scale(scale))\n'
    '    else:\n'
    '        image = _rgb_image(rgb, bands, scale)\n'
    '\n'
    '    feature = []\n'
    '    if mode in ("mle", "ms_mle"):\n'
    '        for channel in np.moveaxis(image, -1, 0):\n'
    '            mean, variance = mle_gaussian(channel)\n'
    '            feature.extend((mean, np.sqrt(variance)))\n'
    '    else:\n'
    '        grid_size = 2 if mode == "grid2" else 4\n'
    '        h, w = image.shape[:2]\n'
    '        if min(h, w) < grid_size:\n'
    '            raise ValueError("grid moments require at least one pixel per cell")\n'
    '\n'
    '        for r0, r1, c0, c1 in grid_cells(h, w, grid_size):\n'
    '            for counts in _channel_counts(image[r0:r1, c0:c1]):\n'
    '                moments = moments_from_counts(counts)\n'
    '                feature.extend((moments["mean"], moments["std"], moments["skewness"]))\n'
    '\n'
    '    return np.asarray(feature, dtype=np.float64)\n'
    '\n'
    '\n'
    'def zscore_fit(train) -> tuple[np.ndarray, np.ndarray]:\n'
    '    """Fit population mean and floored standard deviation on training rows."""\n'
    '    features = np.asarray(train, dtype=np.float64)\n'
    '    if features.ndim != 2 or features.size == 0:\n'
    '        raise ValueError("train must be a nonempty feature matrix")\n'
    '\n'
    '    mean = features.mean(axis=0)\n'
    '    std = np.sqrt(np.mean((features - mean) ** 2, axis=0))\n'
    '    return mean, np.maximum(std, 1e-8)\n'
    '\n'
    '\n'
    'def zscore_apply(features, mean, std) -> np.ndarray:\n'
    '    return (np.asarray(features) - mean) / std\n'
    '\n'
    '\n'
    'def _distance_chunk(query, rows, metric) -> np.ndarray:\n'
    '    if metric == "intersection":\n'
    '        return 1 - np.minimum(query, rows).sum(axis=1)\n'
    '\n'
    '    squared = rows - query\n'
    '    np.square(squared, out=squared)\n'
    '    if metric == "euclid":\n'
    '        return np.sqrt(squared.sum(axis=1))\n'
    '\n'
    '    denominator = rows + query\n'
    '    positive = denominator > 0\n'
    '    np.divide(squared, denominator, out=squared, where=positive)\n'
    '    return np.sum(squared, axis=1, where=positive)\n'
    '\n'
    '\n'
    'def distance_batch(query, database, metric, *, batch_size=2000) -> np.ndarray:\n'
    '    """Compute one distance per database row, using 1 - I for intersection."""\n'
    '    query = np.asarray(query)\n'
    '    database = np.asarray(database)\n'
    '    if query.ndim != 1 or database.ndim != 2 or database.shape[1] != query.size:\n'
    '        raise ValueError("query and database must have the same feature dimension")\n'
    '    if metric not in _PMF_METRICS:\n'
    '        raise ValueError(f"unknown metric: {metric}")\n'
    '    if batch_size <= 0:\n'
    '        raise ValueError("batch_size must be positive")\n'
    '\n'
    '    dtype = np.result_type(query.dtype, database.dtype, np.float32)\n'
    '    query = query.astype(dtype, copy=False)\n'
    '    distances = np.empty(len(database), dtype=dtype)\n'
    '    for start in range(0, len(database), batch_size):\n'
    '        stop = start + batch_size\n'
    '        rows = database[start:stop].astype(dtype, copy=False)\n'
    '        distances[start:stop] = _distance_chunk(query, rows, metric)\n'
    '    return distances\n'
    '\n'
    '\n'
    'def search_topk(query, database, k, metric, *, query_index=None) -> tuple[np.ndarray, np.ndarray]:\n'
    '    """Return sorted database indices and distances, excluding the supplied query index."""\n'
    '    if k <= 0:\n'
    '        raise ValueError("k must be positive")\n'
    '\n'
    '    distances = distance_batch(query, database, metric)\n'
    '    indices = np.arange(len(distances))\n'
    '    if query_index is not None:\n'
    '        indices = indices[indices != query_index]\n'
    '\n'
    '    ranking = np.argsort(distances[indices], kind="stable")[:k]\n'
    '    selected = indices[ranking]\n'
    '    return selected, distances[selected]\n'
    '\n'
    '\n'
    'def precision_at_k(query_class, neighbor_classes) -> float:\n'
    '    labels = np.asarray(neighbor_classes)\n'
    '    if labels.ndim != 1 or labels.size == 0:\n'
    '        raise ValueError("expected at least one neighbor class")\n'
    '    return float(np.mean(labels == query_class))\n'
    '\n'
    '\n'
    'def knn_predict(neighbor_classes, distances) -> int:\n'
    '    """Vote by class, then rank ties by total distance, closest distance, and class index."""\n'
    '    labels = np.asarray(neighbor_classes)\n'
    '    distances = np.asarray(distances, dtype=np.float64)\n'
    '    if labels.ndim != 1 or labels.size == 0 or labels.shape != distances.shape:\n'
    '        raise ValueError("expected matching nonempty neighbor classes and distances")\n'
    '\n'
    '    classes, votes = np.unique(labels, return_counts=True)\n'
    '    contenders = classes[votes == votes.max()]\n'
    '    groups = [distances[labels == label] for label in contenders]\n'
    '    totals = [group.sum() for group in groups]\n'
    '    closest = [group.min() for group in groups]\n'
    '    ranking = np.lexsort((contenders, closest, totals))\n'
    '    return int(contenders[ranking[0]])\n'
    '\n'
    '\n'
    'def confusion_matrix(y_true, y_pred, n_classes) -> np.ndarray:\n'
    '    true = np.asarray(y_true, dtype=np.int64)\n'
    '    predicted = np.asarray(y_pred, dtype=np.int64)\n'
    '    if true.ndim != 1 or true.shape != predicted.shape or n_classes <= 0:\n'
    '        raise ValueError("expected matching label vectors and a positive class count")\n'
    '    if np.any((true < 0) | (true >= n_classes) | (predicted < 0) | (predicted >= n_classes)):\n'
    '        raise ValueError("class labels must be in [0, n_classes)")\n'
    '\n'
    '    cell_indices = true * n_classes + predicted\n'
    '    return np.bincount(cell_indices, minlength=n_classes ** 2).reshape(n_classes, n_classes)\n'
)

In [ ]:
embedded_sources['src/core/color.py'] = (
    'import numpy as np\n'
    '\n'
    '\n'
    'def rgb_to_gray(rgb_u8) -> np.ndarray:\n'
    '    """Apply the specified luminance weights with half-up rounding."""\n'
    '    red, green, blue = np.moveaxis(np.asarray(rgb_u8, dtype=np.int32), -1, 0)\n'
    '    return ((299 * red + 587 * green + 114 * blue + 500) // 1000).astype(np.uint8)\n'
    '\n'
    '\n'
    'def rgb_to_hsv(rgb_u8) -> np.ndarray:\n'
    '    """Convert RGB to HSV with 256 hue bins and byte-valued saturation and value."""\n'
    '    rgb = np.asarray(rgb_u8, dtype=np.int32)\n'
    '    red, green, blue = np.moveaxis(rgb, -1, 0)\n'
    '    value = rgb.max(axis=-1)\n'
    '    chroma = value - rgb.min(axis=-1)\n'
    '\n'
    '    # Express the hue in sixths of a turn with chroma as the denominator.\n'
    '    hue_numerator = np.select(\n'
    '        [red == value, green == value],\n'
    '        [green - blue, blue - red + 2 * chroma],\n'
    '        default=red - green + 4 * chroma,\n'
    '    )\n'
    '    turn = 6 * np.maximum(chroma, 1)\n'
    '    hue = (hue_numerator % turn) * 256 // turn\n'
    '    saturation = (2 * 255 * chroma + value) // (2 * np.maximum(value, 1))\n'
    '    return np.stack((hue, saturation, value), axis=-1).astype(np.uint8)\n'
    '\n'
    '\n'
    'def grid_cells(h, w, g) -> list[tuple[int, int, int, int]]:\n'
    '    """Return row-major cells with exclusive bottom and right edges."""\n'
    '    if g <= 0:\n'
    '        raise ValueError("g must be positive")\n'
    '\n'
    '    row_edges = [i * h // g for i in range(g + 1)]\n'
    '    col_edges = [i * w // g for i in range(g + 1)]\n'
    '    return [\n'
    '        (row_edges[row], row_edges[row + 1], col_edges[col], col_edges[col + 1])\n'
    '        for row in range(g)\n'
    '        for col in range(g)\n'
    '    ]\n'
)

In [ ]:
embedded_sources['src/core/probability.py'] = (
    'from math import erf\n'
    '\n'
    'import numpy as np\n'
    '\n'
    'from src.core.stats import counts_to_pmf\n'
    '\n'
    '\n'
    'def cond_prob(p_a_and_b, p_b) -> float:\n'
    '    if not p_b > 0:\n'
    '        raise ValueError("P(B) must be positive")\n'
    '    return float(p_a_and_b / p_b)\n'
    '\n'
    '\n'
    'def _joint_probabilities(priors, likelihoods) -> np.ndarray:\n'
    '    priors = np.asarray(priors, dtype=np.float64)\n'
    '    likelihoods = np.asarray(likelihoods, dtype=np.float64)\n'
    '    if priors.ndim != 1 or priors.shape != likelihoods.shape:\n'
    '        raise ValueError("priors and likelihoods must be matching vectors")\n'
    '    return priors * likelihoods\n'
    '\n'
    '\n'
    'def total_prob(priors, likelihoods) -> float:\n'
    '    return float(np.sum(_joint_probabilities(priors, likelihoods)))\n'
    '\n'
    '\n'
    'def bayes_posterior(priors, likelihoods) -> tuple[np.ndarray, float]:\n'
    '    joint = _joint_probabilities(priors, likelihoods)\n'
    '    evidence = float(np.sum(joint))\n'
    '    if not evidence > 0:\n'
    '        raise ValueError("evidence must have positive probability")\n'
    '    return joint / evidence, evidence\n'
    '\n'
    '\n'
    'def _class_labels(y, n_classes, n_samples) -> np.ndarray:\n'
    '    labels = np.asarray(y)\n'
    '    if n_classes <= 0 or n_samples == 0 or labels.shape != (n_samples,):\n'
    '        raise ValueError("expected one class label per sample and at least one class and sample")\n'
    '    if not np.issubdtype(labels.dtype, np.integer) or np.any((labels < 0) | (labels >= n_classes)):\n'
    '        raise ValueError("labels must be integers in [0, n_classes)")\n'
    '    return labels.astype(np.int64, copy=False)\n'
    '\n'
    '\n'
    'def discrete_bayes_fit(evidence_bins, y, n_classes, levels, alpha=1.0) -> dict:\n'
    '    bins = np.asarray(evidence_bins)\n'
    '    if bins.ndim != 1 or not np.issubdtype(bins.dtype, np.integer):\n'
    '        raise ValueError("evidence bins must be an integer vector")\n'
    '    if levels <= 0 or np.any((bins < 0) | (bins >= levels)):\n'
    '        raise ValueError("evidence bins must be in [0, levels)")\n'
    '    if not np.isfinite(alpha) or alpha < 0:\n'
    '        raise ValueError("alpha must be finite and nonnegative")\n'
    '\n'
    '    labels = _class_labels(y, n_classes, bins.size)\n'
    '    indices = labels * levels + bins.astype(np.int64, copy=False)\n'
    '    joint_counts = np.bincount(indices, minlength=n_classes * levels).reshape(n_classes, levels)\n'
    '    class_counts = joint_counts.sum(axis=1)\n'
    '    denominator = class_counts + alpha * levels\n'
    '    if np.any(denominator == 0):\n'
    '        raise ValueError("classes without evidence require positive smoothing")\n'
    '    return {\n'
    '        "prior": class_counts / bins.size,\n'
    '        "likelihood": (joint_counts + alpha) / denominator[:, None],\n'
    '        "joint_counts": joint_counts,\n'
    '    }\n'
    '\n'
    '\n'
    'def gaussian_nb_fit(X, y, n_classes, var_floor=1e-6) -> dict:\n'
    '    samples = np.asarray(X, dtype=np.float64)\n'
    '    if samples.ndim != 2 or samples.shape[1] == 0:\n'
    '        raise ValueError("X must be a matrix with at least one feature")\n'
    '    if not np.isfinite(var_floor) or var_floor <= 0:\n'
    '        raise ValueError("var_floor must be finite and positive")\n'
    '\n'
    '    labels = _class_labels(y, n_classes, samples.shape[0])\n'
    '    class_counts = np.bincount(labels, minlength=n_classes)\n'
    '    means = np.zeros((n_classes, samples.shape[1]))\n'
    '    variances = np.full_like(means, var_floor)\n'
    '    for cls in np.flatnonzero(class_counts):\n'
    '        group = samples[labels == cls]\n'
    '        means[cls] = np.mean(group, axis=0)\n'
    '        variances[cls] = np.maximum(np.mean((group - means[cls]) ** 2, axis=0), var_floor)\n'
    '\n'
    '    return {"prior": class_counts / len(samples), "mean": means, "variance": variances}\n'
    '\n'
    '\n'
    'def gaussian_nb_log_posterior(model, X) -> np.ndarray:\n'
    '    samples = np.asarray(X, dtype=np.float64)\n'
    '    means, variances, priors = model["mean"], model["variance"], model["prior"]\n'
    '    if samples.ndim != 2 or samples.shape[1] != means.shape[1]:\n'
    '        raise ValueError("X must have the same feature count as the fitted model")\n'
    '\n'
    '    log_prior = np.full_like(priors, -np.inf)\n'
    '    np.log(priors, where=priors > 0, out=log_prior)\n'
    '    deviations = samples[:, None, :] - means\n'
    '    log_density = -0.5 * np.sum(\n'
    '        np.log(2 * np.pi) + np.log(variances) + deviations ** 2 / variances, axis=-1,\n'
    '    )\n'
    '    scores = log_density + log_prior\n'
    '    shifted = scores - np.max(scores, axis=1, keepdims=True)\n'
    '    return shifted - np.log(np.sum(np.exp(shifted), axis=1, keepdims=True))\n'
    '\n'
    '\n'
    'def chi2_uniform_stat(counts) -> float:\n'
    '    counts = np.asarray(counts, dtype=np.float64)\n'
    '    total = np.sum(counts)\n'
    '    if not np.isfinite(total) or total <= 0 or np.any(counts < 0):\n'
    '        raise ValueError("counts must be nonnegative with a finite, positive total")\n'
    '    expected = total / counts.size\n'
    '    return float(np.sum((counts - expected) ** 2 / expected))\n'
    '\n'
    '\n'
    'def chi2_sf(x, df) -> float:\n'
    '    """Wilson-Hilferty approximation to the chi-square survival function."""\n'
    '    if not df > 0:\n'
    '        raise ValueError("df must be positive")\n'
    '    if x <= 0:\n'
    '        return 1.0\n'
    '\n'
    '    correction = 2 / (9 * df)\n'
    '    z = (np.cbrt(x / df) - (1 - correction)) / np.sqrt(correction)\n'
    '    return (1 - erf(z / np.sqrt(2))) / 2\n'
    '\n'
    '\n'
    'def shannon_entropy(counts) -> float:\n'
    '    pmf = counts_to_pmf(counts)\n'
    '    positive = pmf[pmf > 0]\n'
    '    return float(-np.sum(positive * np.log2(positive)))\n'
    '\n'
    '\n'
    'def pearson_r(x, y) -> float:\n'
    '    x = np.asarray(x, dtype=np.float64).ravel()\n'
    '    y = np.asarray(y, dtype=np.float64).ravel()\n'
    '    if x.shape != y.shape or x.size == 0:\n'
    '        raise ValueError("x and y must have the same positive number of samples")\n'
    '\n'
    '    x = x - np.mean(x)\n'
    '    y = y - np.mean(y)\n'
    '    denominator = np.sqrt(np.sum(x * x)) * np.sqrt(np.sum(y * y))\n'
    '    if denominator == 0:\n'
    '        return float("nan")\n'
    '    return float(np.clip(np.sum(x * y) / denominator, -1, 1))\n'
    '\n'
    '\n'
    'def binom_pmf(n, k, p):\n'
    '    if n < 0 or not 0 <= p <= 1:\n'
    '        raise ValueError("n must be nonnegative and p must be in [0, 1]")\n'
    '\n'
    '    outcomes = np.asarray(k)\n'
    '    pmf = np.zeros(outcomes.shape, dtype=np.float64)\n'
    '    if p == 0 or p == 1:\n'
    '        pmf[...] = outcomes == (n if p == 1 else 0)\n'
    '    else:\n'
    '        supported = (outcomes >= 0) & (outcomes <= n) & (outcomes == np.floor(outcomes))\n'
    '        hits = outcomes[supported].astype(np.int64)\n'
    '        log_factorials = np.concatenate(([0.0], np.cumsum(np.log(np.arange(1, n + 1)))))\n'
    '        log_binomial = log_factorials[n] - log_factorials[hits] - log_factorials[n - hits]\n'
    '        pmf[supported] = np.exp(log_binomial + hits * np.log(p) + (n - hits) * np.log1p(-p))\n'
    '    return float(pmf) if pmf.ndim == 0 else pmf\n'
    '\n'
    '\n'
    'def geometric_pmf(k, p):\n'
    '    if not 0 <= p <= 1:\n'
    '        raise ValueError("p must be in [0, 1]")\n'
    '    outcomes = np.asarray(k, dtype=np.float64)\n'
    '    supported = (outcomes >= 1) & np.isfinite(outcomes) & (outcomes == np.floor(outcomes))\n'
    '    pmf = np.zeros(outcomes.shape, dtype=np.float64)\n'
    '    pmf[supported] = p * (1 - p) ** (outcomes[supported] - 1)\n'
    '    return float(pmf) if pmf.ndim == 0 else pmf\n'
    '\n'
    '\n'
    'def geometric_mean(p):\n'
    '    if not 0 <= p <= 1:\n'
    '        raise ValueError("p must be in [0, 1]")\n'
    '    return float("inf") if p == 0 else float(1 / p)\n'
    '\n'
    '\n'
    'def chebyshev_bound(var, r):\n'
    '    if not var >= 0 or not r > 0:\n'
    '        raise ValueError("variance must be nonnegative and r must be positive")\n'
    '    return float(var / r / r)\n'
)

In [ ]:
embedded_sources['src/core/secret_sharing.py'] = (
    'import numpy as np\n'
    '\n'
    '\n'
    'def _unsigned_array(arr) -> np.ndarray:\n'
    '    array = np.asarray(arr)\n'
    '    if array.dtype not in (np.uint8, np.uint16):\n'
    '        raise ValueError("expected a uint8 or uint16 array")\n'
    '    return array\n'
    '\n'
    '\n'
    'def split_xor(arr, n, rng) -> list[np.ndarray]:\n'
    '    """Draw n-1 uniform shares and close the XOR with the last share."""\n'
    '    source = _unsigned_array(arr)\n'
    '    if n < 2:\n'
    '        raise ValueError("at least two shares are required")\n'
    '\n'
    '    levels = int(np.iinfo(source.dtype).max) + 1\n'
    '    final_share = source.copy()\n'
    '    shares = []\n'
    '    for _ in range(n - 1):\n'
    '        random_share = rng.integers(0, levels, size=source.shape, dtype=source.dtype)\n'
    '        np.bitwise_xor(final_share, random_share, out=final_share)\n'
    '        shares.append(random_share)\n'
    '    shares.append(final_share)\n'
    '    return shares\n'
    '\n'
    '\n'
    'def reconstruct_xor(shares) -> np.ndarray:\n'
    '    """XOR the supplied shares, including a partial set for demonstrations."""\n'
    '    remaining = iter(shares)\n'
    '    first = next(remaining, None)\n'
    '    if first is None:\n'
    '        raise ValueError("at least one share is required")\n'
    '\n'
    '    reconstructed = _unsigned_array(first).copy()\n'
    '    for share in remaining:\n'
    '        share = _unsigned_array(share)\n'
    '        if share.shape != reconstructed.shape or share.dtype != reconstructed.dtype:\n'
    '            raise ValueError("all shares must have the same shape and dtype")\n'
    '        np.bitwise_xor(reconstructed, share, out=reconstructed)\n'
    '    return reconstructed\n'
    '\n'
    '\n'
    'def mse(a, b) -> float:\n'
    '    left = np.asarray(a, dtype=np.float64)\n'
    '    right = np.asarray(b, dtype=np.float64)\n'
    '    if left.shape != right.shape or not left.size:\n'
    '        raise ValueError("expected nonempty arrays of the same shape")\n'
    '\n'
    '    return float(np.mean((left - right) ** 2))\n'
    '\n'
    '\n'
    'def psnr(a, b, max_val) -> float:\n'
    '    if not max_val > 0:\n'
    '        raise ValueError("max_val must be positive")\n'
    '\n'
    '    error = mse(a, b)\n'
    '    if error == 0:\n'
    '        return float("inf")\n'
    '    return float(20 * np.log10(max_val) - 10 * np.log10(error))\n'
    '\n'
    '\n'
    'def flip_bits(arr, p, rng, bits) -> np.ndarray:\n'
    '    """Flip each of the specified bits independently with probability p."""\n'
    '    source = _unsigned_array(arr)\n'
    '    if not 0 <= p <= 1:\n'
    '        raise ValueError("p must be in [0, 1]")\n'
    '    if not 1 <= bits <= np.iinfo(source.dtype).bits:\n'
    '        raise ValueError("bits must be within the array dtype\'s bit width")\n'
    '\n'
    '    flipped = source.copy()\n'
    '    if p == 0:\n'
    '        return flipped\n'
    '    if p == 1:\n'
    '        np.bitwise_xor(flipped, source.dtype.type((1 << bits) - 1), out=flipped)\n'
    '        return flipped\n'
    '\n'
    '    for bit in range(bits):\n'
    '        mask = rng.random(source.shape) < p\n'
    '        np.bitwise_xor(flipped, source.dtype.type(1 << bit), out=flipped, where=mask)\n'
    '    return flipped\n'
    '\n'
    '\n'
    'def _valid_index(value, limit) -> bool:\n'
    '    return isinstance(value, (int, np.integer)) and 0 <= value < limit\n'
    '\n'
    '\n'
    'def _tamper_source(arr, x, y) -> np.ndarray:\n'
    '    source = _unsigned_array(arr)\n'
    '    if source.ndim not in (2, 3):\n'
    '        raise ValueError("expected a 2D or 3D share image")\n'
    '\n'
    '    height, width = source.shape[:2]\n'
    '    if not (_valid_index(x, width) and _valid_index(y, height)):\n'
    '        raise ValueError("pixel coordinates must be inside the share image")\n'
    '    return source\n'
    '\n'
    '\n'
    'def tamper_pixel(arr, x, y, bit=None, channel=None) -> np.ndarray:\n'
    '    """Flip one bit at (x, y), using the MSB and all channels by default."""\n'
    '    source = _tamper_source(arr, x, y)\n'
    '    bits = np.iinfo(source.dtype).bits\n'
    '    if bit is None:\n'
    '        bit = bits - 1\n'
    '    if not _valid_index(bit, bits):\n'
    '        raise ValueError("bit must be within the array dtype\'s bit width")\n'
    '\n'
    '    pixel = (int(y), int(x))\n'
    '    if channel is not None:\n'
    '        if source.ndim != 3 or not _valid_index(channel, source.shape[2]):\n'
    '            raise ValueError("channel must be an index in a 3D share image")\n'
    '        pixel += (int(channel),)\n'
    '\n'
    '    tampered = source.copy()\n'
    '    tampered[pixel] ^= source.dtype.type(1 << int(bit))\n'
    '    return tampered\n'
    '\n'
    '\n'
    'def tamper_block(arr, x, y, size, rng) -> np.ndarray:\n'
    '    """XOR a clipped block at (x, y) with a nonzero mask per element."""\n'
    '    source = _tamper_source(arr, x, y)\n'
    '    if not isinstance(size, (int, np.integer)) or size < 1:\n'
    '        raise ValueError("size must be a positive integer")\n'
    '\n'
    '    height, width = source.shape[:2]\n'
    '    x, y, size = int(x), int(y), int(size)\n'
    '    x_end, y_end = min(width, x + size), min(height, y + size)\n'
    '    tampered = source.copy()\n'
    '    block = tampered[y:y_end, x:x_end]\n'
    '    levels = 1 << np.iinfo(source.dtype).bits\n'
    '    block ^= rng.integers(1, levels, size=block.shape, dtype=source.dtype)\n'
    '    return tampered\n'
)

In [ ]:
embedded_sources['src/core/security_demos.py'] = (
    'import numpy as np\n'
    '\n'
    'from src.core.secret_sharing import reconstruct_xor, split_xor\n'
    '\n'
    '\n'
    'def reuse_key_attack(i1, i2, rng) -> dict[str, np.ndarray]:\n'
    '    """Demonstrate the XOR identity for two arrays encrypted with one pad."""\n'
    '    first, second = np.asarray(i1), np.asarray(i2)\n'
    '    if first.shape != second.shape or first.dtype != second.dtype:\n'
    '        raise ValueError("inputs must have the same shape and dtype")\n'
    '\n'
    '    key, c1 = split_xor(first, 2, rng)\n'
    '    c2 = np.asarray(np.bitwise_xor(second, key))\n'
    '    return {\n'
    '        "key": key,\n'
    '        "c1": c1,\n'
    '        "c2": c2,\n'
    '        "c1_xor_c2": np.asarray(np.bitwise_xor(c1, c2)),\n'
    '        "i1_xor_i2": np.asarray(np.bitwise_xor(first, second)),\n'
    '        "recovered_key": np.asarray(np.bitwise_xor(c1, first)),\n'
    '    }\n'
    '\n'
    '\n'
    'def raid5_parity(disks) -> np.ndarray:\n'
    '    """Return the shared XOR parity block."""\n'
    '    return reconstruct_xor(disks)\n'
    '\n'
    '\n'
    'def raid5_recover(disks, parity, failed) -> np.ndarray:\n'
    '    """Recover one failed data block from the parity and surviving blocks."""\n'
    '    if not 0 <= failed < len(disks):\n'
    '        raise ValueError("failed must be an index in disks")\n'
    '\n'
    '    surviving = (disk for index, disk in enumerate(disks) if index != failed)\n'
    '    return reconstruct_xor((parity, *surviving))\n'
)

In [ ]:
embedded_sources['src/core/spectral.py'] = (
    'import numpy as np\n'
    '\n'
    '\n'
    'def _normalized_difference(bands_u16, positive_band, negative_band) -> np.ndarray:\n'
    '    bands = np.asarray(bands_u16)\n'
    '    positive = bands[..., positive_band].astype(np.float64)\n'
    '    negative = bands[..., negative_band].astype(np.float64)\n'
    '    denominator = positive + negative\n'
    '    return np.divide(\n'
    '        positive - negative,\n'
    '        denominator,\n'
    '        out=np.zeros_like(denominator),\n'
    '        where=denominator != 0,\n'
    '    )\n'
    '\n'
    '\n'
    'def ndvi(bands_u16) -> np.ndarray:\n'
    '    """Return the normalized difference of B08 and B04."""\n'
    '    return _normalized_difference(bands_u16, 7, 3)\n'
    '\n'
    '\n'
    'def ndwi(bands_u16) -> np.ndarray:\n'
    '    """Return the normalized difference of B03 and B08."""\n'
    '    return _normalized_difference(bands_u16, 2, 7)\n'
    '\n'
    '\n'
    'def ndi_to_level(v) -> np.ndarray | np.uint8:\n'
    '    """Map spectral index values from [-1, 1] to levels from 0 to 255."""\n'
    '    clipped = np.clip(np.asarray(v, dtype=np.float64), -1, 1)\n'
    '    return np.floor((clipped + 1) * 127.5 + 0.5).astype(np.uint8)\n'
    '\n'
    '\n'
    'def fit_percentile_scale(samples_u16, lo_q=2, hi_q=98) -> dict:\n'
    '    """Select the lower-ranked percentile value per band across all sample pixels."""\n'
    '    samples = np.asarray(samples_u16)\n'
    '    if samples.shape[-1:] != (13,) or samples.size == 0:\n'
    '        raise ValueError("expected nonempty samples with 13 bands on the last axis")\n'
    '    if not 0 <= lo_q <= hi_q <= 100:\n'
    '        raise ValueError("percentiles must satisfy 0 <= lo_q <= hi_q <= 100")\n'
    '\n'
    '    ordered = np.sort(samples.reshape(-1, 13), axis=0)\n'
    '    last = len(ordered) - 1\n'
    '    lo_idx = int(np.floor(lo_q / 100 * last))\n'
    '    hi_idx = int(np.floor(hi_q / 100 * last))\n'
    '    return {"lo": ordered[lo_idx].tolist(), "hi": ordered[hi_idx].tolist()}\n'
    '\n'
    '\n'
    'def scale_to_8bit(bands_u16, scale) -> np.ndarray:\n'
    '    """Apply the per-band scale, using zero for zero-width intervals."""\n'
    '    bands = np.asarray(bands_u16, dtype=np.int32)\n'
    '    lo, hi = np.asarray([scale["lo"], scale["hi"]])\n'
    '    span = hi - lo\n'
    '    numerator = (bands - lo) * 255\n'
    '    scaled = np.floor_divide(numerator, span, out=np.zeros_like(numerator), where=span != 0)\n'
    '    return np.clip(scaled, 0, 255).astype(np.uint8)\n'
    '\n'
    '\n'
    'def synth_rgb(bands_u16, scale) -> np.ndarray:\n'
    '    """Return scaled B04, B03 and B02 as red, green and blue."""\n'
    '    return scale_to_8bit(bands_u16, scale)[..., [3, 2, 1]]\n'
)

In [ ]:
embedded_sources['src/core/stats.py'] = (
    'import numpy as np\n'
    '\n'
    '\n'
    'def histogram_counts(x, bins: int, levels: int = 256) -> np.ndarray:\n'
    '    """Count integer intensities using equal-width bins over [0, levels)."""\n'
    '    pixels = np.asarray(x)\n'
    '    if bins <= 0 or levels <= 0:\n'
    '        raise ValueError("bins and levels must be positive")\n'
    '    if pixels.size and not np.issubdtype(pixels.dtype, np.integer):\n'
    '        raise ValueError("intensities must be integers")\n'
    '    if np.any(pixels < 0) or np.any(pixels >= levels):\n'
    '        raise ValueError("intensities must be in [0, levels)")\n'
    '\n'
    '    indices = pixels.ravel().astype(np.int64) * bins // levels\n'
    '    return np.bincount(indices, minlength=bins).astype(np.int64, copy=False)\n'
    '\n'
    '\n'
    'def coarsen_counts(counts256, bins: int) -> np.ndarray:\n'
    '    """Sum adjacent groups on the last axis of a full 256-bin histogram."""\n'
    '    counts = np.asarray(counts256)\n'
    '    if counts.shape[-1:] != (256,) or bins <= 0 or 256 % bins:\n'
    '        raise ValueError("expected 256 counts and a bin count that divides 256")\n'
    '    groups = counts.reshape(*counts.shape[:-1], bins, 256 // bins)\n'
    '    return groups.sum(axis=-1, dtype=np.int64)\n'
    '\n'
    '\n'
    'def counts_to_pmf(counts) -> np.ndarray:\n'
    '    counts = np.asarray(counts, dtype=np.float64)\n'
    '    total = np.sum(counts)\n'
    '    if not np.isfinite(total) or total <= 0 or np.any(counts < 0):\n'
    '        raise ValueError("counts must be nonnegative with a finite, positive total")\n'
    '    return counts / total\n'
    '\n'
    '\n'
    'def moments_from_counts(counts, values=None) -> dict:\n'
    '    pmf = counts_to_pmf(counts)\n'
    '    x = np.arange(pmf.size, dtype=np.float64) if values is None else np.asarray(values, dtype=np.float64)\n'
    '    if x.shape != pmf.shape or x.ndim != 1:\n'
    '        raise ValueError("counts and values must be matching vectors")\n'
    '\n'
    '    mean = float(np.sum(x * pmf))\n'
    '    variance = max(0.0, float(np.sum(x * x * pmf)) - mean * mean)\n'
    '    std = float(np.sqrt(variance))\n'
    '    degenerate = variance < 1e-12\n'
    '    skewness = kurtosis = 0.0\n'
    '    if not degenerate:\n'
    '        standardized = (x - mean) / std\n'
    '        skewness = float(np.sum(standardized ** 3 * pmf))\n'
    '        kurtosis = float(np.sum(standardized ** 4 * pmf))\n'
    '\n'
    '    return {\n'
    '        "mean": mean,\n'
    '        "variance": variance,\n'
    '        "std": std,\n'
    '        "skewness": skewness,\n'
    '        "kurtosis": kurtosis,\n'
    '        "degenerate": degenerate,\n'
    '    }\n'
    '\n'
    '\n'
    'def mgf_empirical(values, pmf, t) -> float | np.ndarray:\n'
    '    x = np.asarray(values, dtype=np.float64)\n'
    '    weights = np.asarray(pmf, dtype=np.float64)\n'
    '    if x.ndim != 1 or x.shape != weights.shape:\n'
    '        raise ValueError("values and pmf must be matching vectors")\n'
    '\n'
    '    active = weights != 0\n'
    '    t = np.asarray(t, dtype=np.float64)\n'
    '    result = np.sum(np.exp(t[..., None] * x[active]) * weights[active], axis=-1)\n'
    '    return float(result) if t.ndim == 0 else result\n'
    '\n'
    '\n'
    'def mgf_check(counts256, h=1e-3) -> dict:\n'
    '    pmf = counts_to_pmf(counts256)\n'
    '    if pmf.shape != (256,) or not np.isfinite(h) or h <= 0:\n'
    '        raise ValueError("expected 256 counts and a finite, positive step")\n'
    '\n'
    '    x = np.arange(256, dtype=np.float64) / 255\n'
    '    t = np.array([-h, 0.0, h])\n'
    '    m = mgf_empirical(x, pmf, t)\n'
    '    minus, center, plus = map(float, m)\n'
    '    return {\n'
    '        "t": t.tolist(),\n'
    '        "m": m.tolist(),\n'
    '        "m1_fd": (plus - minus) / (2 * h),\n'
    '        "m1_exact": float(np.sum(x * pmf)),\n'
    '        "m2_fd": (plus - 2 * center + minus) / h ** 2,\n'
    '        "m2_exact": float(np.sum(x * x * pmf)),\n'
    '        "scale_note": "Level intensitas diskala dengan x/255.",\n'
    '    }\n'
    '\n'
    '\n'
    'def _samples(x) -> np.ndarray:\n'
    '    samples = np.asarray(x, dtype=np.float64)\n'
    '    if samples.size == 0:\n'
    '        raise ValueError("MLE requires at least one sample")\n'
    '    return samples\n'
    '\n'
    '\n'
    'def mle_gaussian(x) -> tuple[float, float]:\n'
    '    samples = _samples(x)\n'
    '    mean = np.mean(samples)\n'
    '    variance = np.mean((samples - mean) ** 2)\n'
    '    return float(mean), float(variance)\n'
    '\n'
    '\n'
    'def mle_bernoulli(x) -> float:\n'
    '    return float(np.mean(_samples(x)))\n'
    '\n'
    '\n'
    'def mle_poisson(x) -> float:\n'
    '    return float(np.mean(_samples(x)))\n'
    '\n'
    '\n'
    'def mle_exponential(x) -> float:\n'
    '    """Estimate the exponential scale (mean waiting time)."""\n'
    '    return float(np.mean(_samples(x)))\n'
    '\n'
    '\n'
    'def mle_geometric(x) -> float:\n'
    '    samples = _samples(x)\n'
    '    if np.any(samples < 1):\n'
    '        raise ValueError("geometric samples start at 1")\n'
    '    return float(samples.size / np.sum(samples))\n'
)

In [ ]:
embedded_sources['src/data/env.py'] = (
    'import os\n'
    'from dataclasses import dataclass\n'
    'from pathlib import Path\n'
    '\n'
    'from dotenv import load_dotenv\n'
    '\n'
    '\n'
    'PROJECT_ROOT = Path(__file__).resolve().parents[2]\n'
    '\n'
    '\n'
    '@dataclass(frozen=True)\n'
    'class Settings:\n'
    '    rgb_dir: Path | None\n'
    '    allbands_dir: Path | None\n'
    '    app_mode: str\n'
    '    artifacts_dir: Path\n'
    '    seed: int\n'
    '\n'
    '\n'
    'def load_settings() -> Settings:\n'
    '    load_dotenv(PROJECT_ROOT / ".env", override=False)\n'
    '    rgb_dir = os.getenv("EUROSAT_RGB_DIR")\n'
    '    allbands_dir = os.getenv("EUROSAT_ALLBANDS_DIR")\n'
    '    artifacts_dir = os.getenv("ARTIFACTS_DIR") or "artifacts"\n'
    '\n'
    '    return Settings(\n'
    '        rgb_dir=PROJECT_ROOT / Path(rgb_dir).expanduser() if rgb_dir else None,\n'
    '        allbands_dir=PROJECT_ROOT / Path(allbands_dir).expanduser() if allbands_dir else None,\n'
    '        app_mode=os.getenv("APP_MODE") or "dev",\n'
    '        artifacts_dir=PROJECT_ROOT / Path(artifacts_dir).expanduser(),\n'
    '        seed=int(os.getenv("SEED") or "42"),\n'
    '    )\n'
)

embedded_sources['src/data/loader.py'] = (
    'from pathlib import Path\n'
    '\n'
    'import numpy as np\n'
    'import tifffile\n'
    'from PIL import Image\n'
    '\n'
    '\n'
    'def load_rgb(path: str | Path) -> np.ndarray:\n'
    '    """Load a JPEG as an RGB uint8 array."""\n'
    '    with Image.open(path) as image:\n'
    '        return np.array(image.convert("RGB"), dtype=np.uint8)\n'
    '\n'
    '\n'
    'def load_tif(path: str | Path) -> np.ndarray:\n'
    '    """Load a multispectral TIFF with the 13 bands on the last axis."""\n'
    '    bands = tifffile.imread(path)\n'
    '    band_axes = [axis for axis, size in enumerate(bands.shape) if size == 13]\n'
    '    if bands.ndim != 3 or len(band_axes) != 1:\n'
    '        raise ValueError(f"{path}: expected one 13-band axis in a 3D TIFF, got {bands.shape}")\n'
    '\n'
    '    if bands.dtype.kind != "u" or bands.dtype.itemsize != 2:\n'
    '        raise ValueError(f"{path}: expected a uint16 TIFF, got {bands.dtype}")\n'
    '\n'
    '    return np.moveaxis(bands, band_axes[0], -1).astype(np.uint16, copy=False)\n'
)

embedded_sources['src/data/manifest.py'] = (
    'import csv\n'
    'import re\n'
    'import sys\n'
    'from pathlib import Path\n'
    'from tempfile import NamedTemporaryFile\n'
    'from typing import TypedDict\n'
    '\n'
    'import numpy as np\n'
    '\n'
    'from src.data.env import load_settings\n'
    '\n'
    '\n'
    'CLASSES = (\n'
    '    "AnnualCrop", "Forest", "HerbaceousVegetation", "Highway", "Industrial",\n'
    '    "Pasture", "PermanentCrop", "Residential", "River", "SeaLake",\n'
    ')\n'
    '\n'
    '\n'
    'class ManifestRow(TypedDict):\n'
    '    id: str\n'
    '    cls: str\n'
    '    cls_idx: int\n'
    '    split: str\n'
    '    rgb_path: str\n'
    '    tif_path: str\n'
    '    in_gallery: bool\n'
    '    in_colab_subset: bool\n'
    '\n'
    '\n'
    'def scan_images(root: Path, extension: str) -> tuple[dict[str, str], list[str]]:\n'
    '    if not root.is_dir():\n'
    '        raise ValueError(f"Direktori dataset tidak ada: {root}")\n'
    '\n'
    '    paths = {}\n'
    '    invalid = []\n'
    '    for cls in CLASSES:\n'
    '        id_pattern = re.compile(rf"{cls}_[0-9]+")\n'
    '        for path in sorted((root / cls).glob(f"*{extension}")):\n'
    '            if not path.is_file():\n'
    '                continue\n'
    '            relative_path = path.relative_to(root).as_posix()\n'
    '            if id_pattern.fullmatch(path.stem):\n'
    '                paths[path.stem] = relative_path\n'
    '            else:\n'
    '                invalid.append(relative_path)\n'
    '    return paths, invalid\n'
    '\n'
    '\n'
    'def scan_dataset(\n'
    '    rgb_dir: Path, tif_dir: Path, *, report_unmatched: bool = False,\n'
    ') -> tuple[dict[str, str], dict[str, str]]:\n'
    '    rgb, invalid_rgb = scan_images(rgb_dir, ".jpg")\n'
    '    tif, invalid_tif = scan_images(tif_dir, ".tif")\n'
    '    if report_unmatched:\n'
    '        for modality, ids in (("JPG", rgb.keys() - tif.keys()), ("TIFF", tif.keys() - rgb.keys())):\n'
    '            print(f"ID hanya di {modality} ({len(ids)}): {\', \'.join(sorted(ids))}")\n'
    '        invalid = invalid_rgb + invalid_tif\n'
    '        if invalid:\n'
    '            examples = ", ".join(invalid[:5])\n'
    '            print(f"{len(invalid)} berkas di luar pola ID EuroSAT dilewati. Contoh: {examples}")\n'
    '    if not rgb and not tif:\n'
    '        raise ValueError("Tidak ada ID EuroSAT di direktori dataset.")\n'
    '    return rgb, tif\n'
    '\n'
    '\n'
    'def select_ids(\n'
    '    candidates: list[str], count: int, rng: np.random.Generator, cls: str, split: str,\n'
    ') -> set[str]:\n'
    '    if len(candidates) < count:\n'
    '        raise ValueError(\n'
    '            f"{cls} ({split}): perlu {count} pasangan JPG/TIFF, tersedia {len(candidates)}."\n'
    '        )\n'
    '    return set(rng.choice(candidates, size=count, replace=False).tolist())\n'
    '\n'
    '\n'
    'def manifest_rows(rgb: dict[str, str], tif: dict[str, str], seed: int) -> list[ManifestRow]:\n'
    '    all_ids = sorted(rgb.keys() | tif.keys())\n'
    '    paired_ids = rgb.keys() & tif.keys()\n'
    '    rows: list[ManifestRow] = []\n'
    '    for cls_idx, cls in enumerate(CLASSES):\n'
    '        ids = [image_id for image_id in all_ids if image_id.startswith(f"{cls}_")]\n'
    '        if not ids:\n'
    '            continue\n'
    '\n'
    '        rng = np.random.default_rng(seed)\n'
    '        train_size = len(ids) * 4 // 5\n'
    '        train = {ids[i] for i in rng.permutation(len(ids))[:train_size]}\n'
    '        train_paired = [image_id for image_id in ids if image_id in paired_ids and image_id in train]\n'
    '        test_paired = [\n'
    '            image_id for image_id in ids if image_id in paired_ids and image_id not in train\n'
    '        ]\n'
    '        gallery = select_ids(train_paired, 10, rng, cls, "train") | select_ids(\n'
    '            test_paired, 15, rng, cls, "test"\n'
    '        )\n'
    '        colab = select_ids(train_paired, 160, rng, cls, "train") | select_ids(\n'
    '            test_paired, 40, rng, cls, "test"\n'
    '        )\n'
    '\n'
    '        for image_id in ids:\n'
    '            rows.append({\n'
    '                "id": image_id,\n'
    '                "cls": cls,\n'
    '                "cls_idx": cls_idx,\n'
    '                "split": "train" if image_id in train else "test",\n'
    '                "rgb_path": rgb.get(image_id, ""),\n'
    '                "tif_path": tif.get(image_id, ""),\n'
    '                "in_gallery": image_id in gallery,\n'
    '                "in_colab_subset": image_id in colab,\n'
    '            })\n'
    '    return rows\n'
    '\n'
    '\n'
    'def build_manifest(rgb_dir: Path, tif_dir: Path, seed: int = 42) -> list[ManifestRow]:\n'
    '    """Build rows with one split assignment per EuroSAT ID."""\n'
    '    rgb, tif = scan_dataset(rgb_dir, tif_dir)\n'
    '    return manifest_rows(rgb, tif, seed)\n'
    '\n'
    '\n'
    'def write_manifest(rgb_dir: Path, tif_dir: Path, destination: Path, seed: int = 42) -> None:\n'
    '    """Report skipped files and unmatched IDs, then write the manifest CSV."""\n'
    '    rgb, tif = scan_dataset(rgb_dir, tif_dir, report_unmatched=True)\n'
    '    rows = manifest_rows(rgb, tif, seed)\n'
    '    destination.parent.mkdir(parents=True, exist_ok=True)\n'
    '    temporary = NamedTemporaryFile(\n'
    '        mode="w", newline="", encoding="utf-8", dir=destination.parent, delete=False,\n'
    '    )\n'
    '    temporary_path = Path(temporary.name)\n'
    '    try:\n'
    '        with temporary as stream:\n'
    '            writer = csv.DictWriter(stream, fieldnames=list(ManifestRow.__annotations__))\n'
    '            writer.writeheader()\n'
    '            writer.writerows(rows)\n'
    '        temporary_path.replace(destination)\n'
    '    finally:\n'
    '        temporary_path.unlink(missing_ok=True)\n'
    '\n'
    '\n'
    'def main() -> int:\n'
    '    try:\n'
    '        settings = load_settings()\n'
    '        if settings.rgb_dir is None or settings.allbands_dir is None:\n'
    '            raise ValueError("Isi EUROSAT_RGB_DIR dan EUROSAT_ALLBANDS_DIR di .env.")\n'
    '        destination = settings.artifacts_dir / "manifest.csv"\n'
    '        write_manifest(settings.rgb_dir, settings.allbands_dir, destination, settings.seed)\n'
    '    except (OSError, ValueError) as exc:\n'
    '        print(exc, file=sys.stderr)\n'
    '        return 1\n'
    '\n'
    '    print(f"Manifest tersimpan di {destination}")\n'
    '    return 0\n'
    '\n'
    '\n'
    'if __name__ == "__main__":\n'
    '    raise SystemExit(main())\n'
)

embedded_sources['src/experiments/c1_bins.py'] = (
    'import numpy as np\n'
    '\n'
    'from src.core.cbir import counts_to_feature, extract_counts\n'
    'from src.experiments.common import BINS, result, table\n'
    '\n'
    '\n'
    'def run(data, evaluation):\n'
    '    rng = np.random.default_rng(data.seed)\n'
    '    selection = rng.choice(len(data.rows), min(200, len(data.rows)), replace=False)\n'
    '    images = [data.rgb(i) for i in selection]\n'
    '    rows = []\n'
    '    for bins in BINS:\n'
    '        entry, search_seconds = evaluation("rgb", "chi2", bins)\n'
    '        start = data.clock()\n'
    '        for image in images:\n'
    '            counts_to_feature(extract_counts("rgb", rgb=image), bins)\n'
    '        extract_ms = (data.clock() - start) * 1000 / len(images)\n'
    '        rows.append([bins, bins * 3, entry["macro_precision_at_k"], extract_ms, search_seconds * 1000])\n'
    '    best = max(rows, key=lambda row: row[2])\n'
    '    chart = {"kind": "line", "title": "Precision@5 menurut jumlah bin", "x": list(BINS),\n'
    '             "series": [{"name": "Precision@5", "y": [row[2] for row in rows]}]}\n'
    '    return result("C1 — Pengaruh bin", "Pencarian histogram RGB dengan chi-square.", data,\n'
    '                  [table(["Bin", "Dimensi", "P@5", "Ekstraksi ms", "Pencarian ms"], rows)],\n'
    '                  [f"Bin {best[0]} mencapai P@5 tertinggi sebesar {best[2]:.4f}."],\n'
    '                  charts=[chart], params={"bins": list(BINS), "metric": "chi2"},\n'
    '                  n_queries=entry["protocol"]["n_queries"])\n'
)

embedded_sources['src/experiments/c2_metrics.py'] = (
    'import numpy as np\n'
    '\n'
    'from src.core.cbir import confusion_matrix\n'
    'from src.core.probability import discrete_bayes_fit, gaussian_nb_fit, gaussian_nb_log_posterior\n'
    'from src.data.manifest import CLASSES\n'
    'from src.experiments.common import heatmap, result, table\n'
    '\n'
    '\n'
    'def run(data, evaluation):\n'
    '    classes = data.classes\n'
    '    methods = [("rgb", "euclid"), ("rgb", "chi2"), ("rgb", "intersection"), ("mle", "euclid")]\n'
    '    rows, per_class, charts = [], [], []\n'
    '    for mode, metric in methods:\n'
    '        entry, _ = evaluation(mode, metric, 32)\n'
    '        name = f"{mode} {metric}"\n'
    '        rows.append([name, entry["macro_precision_at_k"], entry["knn_accuracy"]])\n'
    '        per_class.append([name, *(item["precision_at_k"] for item in entry["per_class"])])\n'
    '        if metric == "chi2":\n'
    '            charts.append(heatmap(name, entry))\n'
    '\n'
    '    db = data.split_rows(0, classes)\n'
    '    queries = data.sample(1, classes, None if data.all_queries else 50)\n'
    '    labels = np.searchsorted(classes, data.labels)\n'
    '    for mode in ("mle", "grid2"):\n'
    '        features = data.index[f"feat_{mode}"]\n'
    '        model = gaussian_nb_fit(features[db], labels[db], len(classes))\n'
    '        predicted = gaussian_nb_log_posterior(model, features[queries]).argmax(axis=1)\n'
    '        rows.append([f"GNB {mode}", None, float(np.mean(predicted == labels[queries]))])\n'
    '        if mode == "mle":\n'
    '            charts.append(heatmap("GNB mle", {\n'
    '                "classes": [CLASSES[cls] for cls in classes],\n'
    '                "confusion_matrix": confusion_matrix(labels[queries], predicted, len(classes)).tolist(),\n'
    '            }))\n'
    '\n'
    '    counts = data.index["counts_gray"]\n'
    '    intensity_sum = counts @ np.arange(256, dtype=np.int64)\n'
    '    pixel_count = counts.sum(axis=1, dtype=np.int64)\n'
    '    bins = (8 * intensity_sum) // (256 * pixel_count)\n'
    '    model = discrete_bayes_fit(bins[db], labels[db], len(classes), 8)\n'
    '    predicted = (model["prior"][:, None] * model["likelihood"][:, bins[queries]]).argmax(axis=0)\n'
    '    rows.append(["Bayes mean_gray", None, float(np.mean(predicted == labels[queries]))])\n'
    '    best = max(rows, key=lambda row: row[2])\n'
    '    return result("C2 — Metode pencarian dan klasifikasi", "Perbandingan kNN, GNB dan Bayes diskret.", data,\n'
    '                  [table(["Metode", "P@5", "Akurasi"], rows),\n'
    '                   table(["Metode", *(CLASSES[cls] for cls in classes)], per_class)],\n'
    '                  [f"Akurasi tertinggi {best[2]:.4f} diperoleh {best[0]}."],\n'
    '                  charts=charts, params={"bins": 32, "bayes_levels": 8}, n_queries=len(queries))\n'
)

embedded_sources['src/experiments/c3_bitflip.py'] = (
    'import numpy as np\n'
    '\n'
    'from src.core.color import rgb_to_gray\n'
    'from src.core.probability import geometric_pmf\n'
    'from src.core.secret_sharing import flip_bits, reconstruct_xor, split_xor\n'
    'from src.experiments.common import result, table\n'
    '\n'
    'PROBABILITIES = (0.0005, 0.001, 0.005, 0.01, 0.02, 0.05, 0.1)\n'
    '\n'
    '\n'
    'def _trial_counts(images, trials, bits, damaged, p, rng):\n'
    '    error_counts, first_errors = [], []\n'
    '    for image in images:\n'
    '        clean = np.broadcast_to(image, (trials, *image.shape))\n'
    '        shares = split_xor(clean, 3, rng)\n'
    '        for i in range(damaged):\n'
    '            shares[i] = flip_bits(shares[i], p, rng, bits)\n'
    '        wrong = (reconstruct_xor(shares) != clean).reshape(trials, -1)\n'
    '        error_counts.extend(wrong.sum(axis=1))\n'
    '        first_errors.extend(np.where(wrong.any(axis=1), wrong.argmax(axis=1) + 1, 0))\n'
    '    return np.array(error_counts), np.array(first_errors)\n'
    '\n'
    '\n'
    'def run(data):\n'
    '    sample = data.sample(1, data.classes, 2)\n'
    '    images = {\n'
    '        8: [rgb_to_gray(data.rgb(i)) for i in sample],\n'
    '        16: [data.bands(i)[..., 3] for i in sample],\n'
    '    }\n'
    '    rng = np.random.default_rng(data.seed)\n'
    '    trials = 3 if data.smoke else 30\n'
    '    rows, geometric, chebyshev = [], [], []\n'
    '    series = []\n'
    '    for bits, sources in images.items():\n'
    '        pixels = sources[0].size\n'
    '        for damaged in (1, 2):\n'
    '            empirical = []\n'
    '            for p in PROBABILITIES:\n'
    '                effective = (1 - (1 - 2 * p) ** damaged) / 2\n'
    '                expected = 1 - (1 - effective) ** bits\n'
    '                counts, first = _trial_counts(sources, trials, bits, damaged, p, rng)\n'
    '                observed = float(counts.mean() / pixels)\n'
    '                empirical.append(observed)\n'
    '                rows.append([bits, damaged, p, effective, observed, expected, abs(observed - expected)])\n'
    '                if p == 0.01:\n'
    '                    for position in range(1, min(pixels, 64) + 1):\n'
    '                        geometric.append([bits, damaged, position, float(np.mean(first == position)),\n'
    '                                          geometric_pmf(position, expected)])\n'
    '                    mu = pixels * expected\n'
    '                    sigma = np.sqrt(pixels * expected * (1 - expected))\n'
    '                    for k in range(1, 6):\n'
    '                        tail = float(np.mean(np.abs(counts - mu) >= k * sigma))\n'
    '                        chebyshev.append([bits, damaged, k, tail, 1 / k ** 2])\n'
    '            series.append({"name": f"{bits} bit, {damaged} share", "y": empirical})\n'
    '\n'
    '    worst = max(rows, key=lambda row: row[-1])\n'
    '    return result("C3 — Galat bit-flip", "Simulasi galat piksel dan posisi galat pertama.", data,\n'
    '                  [table(["Bit", "Share rusak", "p", "p efektif", "Empiris", "Teori", "Selisih"], rows),\n'
    '                   table(["Bit", "Share rusak", "Posisi", "Empiris", "Geometrik"], geometric),\n'
    '                   table(["Bit", "Share rusak", "k sigma", "Empiris", "Batas Chebyshev"], chebyshev)],\n'
    '                  [f"Selisih empiris–teori maksimum {worst[-1]:.4f} pada {worst[0]} bit dan p={worst[2]}.",\n'
    '                   "PMF empiris menghitung semua percobaan, termasuk percobaan tanpa piksel salah."],\n'
    '                  charts=[{"kind": "line", "title": "Galat piksel", "x": list(PROBABILITIES), "series": series}],\n'
    '                  params={"p": list(PROBABILITIES), "trials": trials}, n_queries=len(sample))\n'
)

embedded_sources['src/experiments/c4a_modalities.py'] = (
    'from src.core.cbir import MODE_INFO\n'
    'from src.data.manifest import CLASSES\n'
    'from src.experiments.common import evaluate, heatmap, result, table\n'
    '\n'
    'CLASSES_C4A = ("Forest", "Pasture", "HerbaceousVegetation", "AnnualCrop")\n'
    '\n'
    '\n'
    'def run(data, modes):\n'
    '    classes = sorted(set(data.classes) & {CLASSES.index(name) for name in CLASSES_C4A})\n'
    '    entries = [\n'
    '        evaluate(data, mode, metric, 32, classes)[0]\n'
    '        for mode in modes\n'
    '        for metric in (("chi2", "euclid") if MODE_INFO[mode]["kind"] == "pmf" else ("euclid",))\n'
    '    ]\n'
    '    best = max(entries, key=lambda entry: entry["macro_precision_at_k"])\n'
    '    rows = [[i["mode"], i["metric"], i["macro_precision_at_k"], i["knn_accuracy"]] for i in entries]\n'
    '    return result("C4a — Modalitas pada kelas mirip", "Perbandingan mode pada kelas terpilih.", data,\n'
    '                  [table(["Mode", "Metrik", "P@5", "Akurasi"], rows)],\n'
    '                  [f"{best[\'mode\']} dengan {best[\'metric\']} mencapai P@5 {best[\'macro_precision_at_k\']:.4f}.",\n'
    '                   f"Peluang acak untuk {len(classes)} kelas adalah {1 / len(classes):.4f}."],\n'
    '                  charts=[heatmap(f"{best[\'mode\']} {best[\'metric\']}", best)],\n'
    '                  params={"modes": list(modes), "bins": 32}, n_queries=best["protocol"]["n_queries"])\n'
)

embedded_sources['src/experiments/c4b_partial_shares.py'] = (
    'from collections import defaultdict\n'
    '\n'
    'import numpy as np\n'
    '\n'
    'from src.core.cbir import counts_to_feature, extract_counts, search_topk\n'
    'from src.core.color import rgb_to_gray\n'
    'from src.core.probability import chi2_sf, chi2_uniform_stat, pearson_r, shannon_entropy\n'
    'from src.core.secret_sharing import mse, reconstruct_xor, split_xor, tamper_pixel\n'
    'from src.core.stats import histogram_counts\n'
    'from src.experiments.common import features, result, table\n'
    '\n'
    '\n'
    'def run(data, mode, bins=32, metric="chi2"):\n'
    '    sample = data.sample(1, data.classes, 2)\n'
    '    db = data.split_rows(0, data.classes)\n'
    '    database, _ = features(data.index, mode, bins, db, sample)\n'
    '    rng = np.random.default_rng(data.seed)\n'
    '    records = defaultdict(list)\n'
    '    tamper = []\n'
    '    for i in sample:\n'
    '        image = data.rgb(i)\n'
    '        source = rgb_to_gray(image) if mode == "gray" else image\n'
    '        for n in (2, 3, 4, 8):\n'
    '            shares = split_xor(source, n, rng)\n'
    '            for k in range(1, n + 1):\n'
    '                subsets = [("awal", shares[:k])]\n'
    '                if k < n:\n'
    '                    subsets.append(("akhir", shares[-k:]))\n'
    '                for label, subset in subsets:\n'
    '                    partial = reconstruct_xor(subset)\n'
    '                    counts = histogram_counts(partial, 256)\n'
    '                    image_for_feature = np.repeat(partial[..., None], 3, axis=2) if mode == "gray" else partial\n'
    '                    query = counts_to_feature(extract_counts(mode, rgb=image_for_feature), bins)\n'
    '                    hits, _ = search_topk(query, database, min(5, len(db)), metric)\n'
    '                    values = (shannon_entropy(counts), chi2_sf(chi2_uniform_stat(counts), 255),\n'
    '                              pearson_r(source, partial), mse(source, partial),\n'
    '                              float(np.mean(data.labels[db[hits]] == data.labels[i])))\n'
    '                    records[n, k, label].append(values)\n'
    '            changed = reconstruct_xor([tamper_pixel(shares[0], 0, 0, bit=2), *shares[1:]])\n'
    '            delta = np.abs(changed.astype(np.int16) - source)\n'
    '            pixels = np.any(delta != 0, axis=-1) if source.ndim == 3 else delta != 0\n'
    '            tamper.append([str(data.index["ids"][i]), n, int(pixels.sum()), int(delta.max())])\n'
    '\n'
    '    rows = []\n'
    '    for (n, k, subset), values in sorted(records.items()):\n'
    '        columns = np.asarray(values).T\n'
    '        means = [float(np.mean(column[np.isfinite(column)])) if np.isfinite(column).any() else None\n'
    '                 for column in columns]\n'
    '        rows.append([n, k, subset, *means, float(np.mean(columns[1] > 0.05))])\n'
    '    partial_rows = [row for row in rows if row[1] < row[0]]\n'
    '    best = max(partial_rows, key=lambda row: row[7])\n'
    '    return result("C4b — Share parsial", "Evaluasi statistik share dan uji perubahan piksel.", data,\n'
    '                  [table(["n", "k", "Himpunan", "Entropi", "p-value", "Pearson", "MSE", "P@5",\n'
    '                          "Proporsi p-value > 0,05"], rows),\n'
    '                   table(["ID", "n", "Piksel berubah", "Selisih maksimum"], tamper)],\n'
    '                  [f"P@5 parsial tertinggi {best[7]:.4f}; peluang acak {1 / len(data.classes):.4f}.",\n'
    '                   f"Rata-rata proporsi p-value > 0,05 untuk share parsial adalah "\n'
    '                   f"{np.mean([row[8] for row in partial_rows]):.4f}."],\n'
    '                  params={"n": [2, 3, 4, 8], "mode": mode, "bins": bins, "metric": metric},\n'
    '                  n_queries=len(sample))\n'
)

embedded_sources['src/experiments/c4c_complexity.py'] = (
    'import numpy as np\n'
    '\n'
    'from src.core.cbir import search_topk\n'
    'from src.experiments.common import features, result, table\n'
    '\n'
    '\n'
    'def run(data):\n'
    '    rng = np.random.default_rng(data.seed)\n'
    '    n = len(data.rows)\n'
    '    sizes = [10, 20, n] if data.smoke else [100, 300, 1000, 3000, 10000, 27000]\n'
    '    sizes = sorted({min(size, n) for size in sizes})\n'
    '    order = rng.permutation(n)\n'
    '    query_ids = data.sample(1, data.classes, 5)\n'
    '    rows = []\n'
    '    for size in sizes:\n'
    '        indices = order[:size]\n'
    '        database, queries = features(data.index, "rgb", 32, indices, query_ids)\n'
    '        start = data.clock()\n'
    '        for query in queries:\n'
    '            search_topk(query, database, 5, "chi2")\n'
    '        milliseconds = (data.clock() - start) * 1000 / len(queries)\n'
    '        ids = data.index["ids"][indices].tolist()\n'
    '        targets = rng.choice(ids, size=1000)\n'
    '        comparisons = float(np.mean([ids.index(target) + 1 for target in targets]))\n'
    '        rows.append([size, milliseconds, comparisons, (size + 1) / 2])\n'
    '\n'
    '    x, y, _, _ = np.asarray(rows).T\n'
    '    dx, dy = x - x.mean(), y - y.mean()\n'
    '    spread = float(dx @ dx)\n'
    '    slope = float((dx @ dy) / spread) if spread else None\n'
    '    residuals = dy - slope * dx if slope is not None else dy\n'
    '    total = float(np.sum(dy ** 2))\n'
    '    r_squared = 1 - float(np.sum(residuals ** 2)) / total if total and slope is not None else None\n'
    '    charts = [{"kind": "line", "title": "Waktu pencarian", "x": sizes,\n'
    '               "series": [{"name": "ms/query", "y": y.tolist()}]}]\n'
    '    findings = [f"Pada N={sizes[-1]}, rata-rata scan {rows[-1][2]:.1f} perbandingan; teori {rows[-1][3]:.1f}."]\n'
    '    if r_squared is not None:\n'
    '        findings.append(f"Regresi waktu memberi kemiringan {slope:.6f} ms/citra dan R²={r_squared:.4f}.")\n'
    '    return result("C4c — Kompleksitas", "Skala waktu CBIR dan scan pencocokan eksak.", data,\n'
    '                  [table(["Database", "ms/query", "Scan empiris", "Scan teori"], rows),\n'
    '                   table(["Kemiringan", "R²"], [[slope, r_squared]])],\n'
    '                  findings, charts=charts, params={"trials": 1000, "sizes": sizes},\n'
    '                  n_queries=len(query_ids))\n'
)

embedded_sources['src/experiments/common.py'] = (
    '"""Shared offline evaluation protocol and artifact formatting."""\n'
    'from dataclasses import dataclass\n'
    '\n'
    'import numpy as np\n'
    '\n'
    'from src.core.cbir import MODE_INFO, confusion_matrix, knn_predict, search_topk, zscore_fit\n'
    'from src.core.stats import coarsen_counts\n'
    'from src.data.loader import load_rgb, load_tif\n'
    'from src.data.manifest import CLASSES\n'
    '\n'
    'BINS = (8, 16, 32, 64, 128, 256)\n'
    'COUNT_KEYS = {\n'
    '    "gray": ("counts_gray",), "rgb": ("counts_rgb",), "hsv": ("counts_hsv",),\n'
    '    "ms13": ("counts_ms",), "ndvi_ndwi": ("counts_ndvi", "counts_ndwi"),\n'
    '    "ms13_idx": ("counts_ms", "counts_ndvi", "counts_ndwi"),\n'
    '}\n'
    '\n'
    '\n'
    '@dataclass\n'
    'class Dataset:\n'
    '    index: dict\n'
    '    rows: list\n'
    '    rgb_root: object\n'
    '    tif_root: object\n'
    '    clock: object\n'
    '    seed: int = 42\n'
    '    smoke: bool = False\n'
    '    all_queries: bool = False\n'
    '\n'
    '    @property\n'
    '    def labels(self):\n'
    '        return self.index["cls_idx"]\n'
    '\n'
    '    @property\n'
    '    def classes(self):\n'
    '        return np.unique(self.labels)\n'
    '\n'
    '    def rgb(self, i):\n'
    '        return load_rgb(self.rgb_root / self.rows[i]["rgb_path"])\n'
    '\n'
    '    def bands(self, i):\n'
    '        return load_tif(self.tif_root / self.rows[i]["tif_path"])\n'
    '\n'
    '    def split_rows(self, split, classes):\n'
    '        return np.flatnonzero((self.index["split"] == split) & np.isin(self.labels, classes))\n'
    '\n'
    '    def sample(self, split, classes, per_class):\n'
    '        rng = np.random.default_rng(self.seed)\n'
    '        selected = []\n'
    '        for cls in classes:\n'
    '            rows = self.split_rows(split, [cls])\n'
    '            if per_class is not None:\n'
    '                rows = rng.choice(rows, min(per_class, len(rows)), replace=False)\n'
    '            selected.extend(sorted(rows))\n'
    '        return np.array(selected, dtype=np.int64)\n'
    '\n'
    '\n'
    'def features(index, mode, bins, db, queries, *, refit=False):\n'
    '    if mode in COUNT_KEYS:\n'
    '        def pmfs(selection):\n'
    '            channels = []\n'
    '            for key in COUNT_KEYS[mode]:\n'
    '                counts = index[key][selection]\n'
    '                coarse = coarsen_counts(counts, bins).reshape(len(selection), -1, bins).astype(np.float32)\n'
    '                coarse /= coarse.sum(axis=-1, keepdims=True)\n'
    '                channels.append(coarse)\n'
    '            combined = np.concatenate(channels, axis=1)\n'
    '            return (combined / combined.shape[1]).reshape(len(selection), -1)\n'
    '\n'
    '        return pmfs(db), pmfs(queries)\n'
    '\n'
    '    rows = index[f"feat_{mode}"]\n'
    '    if refit:\n'
    '        mean, std = zscore_fit(rows[db])\n'
    '    else:\n'
    '        mean, std = index[f"z_{mode}_mean"], index[f"z_{mode}_std"]\n'
    '    return (rows[db] - mean) / std, (rows[queries] - mean) / std\n'
    '\n'
    '\n'
    'def evaluate(data, mode, metric, bins, classes=None):\n'
    '    refit = classes is not None\n'
    '    if classes is None:\n'
    '        classes = data.classes\n'
    '    db = data.split_rows(0, classes)\n'
    '    queries = data.sample(1, classes, None if data.all_queries else 50)\n'
    '    database, query_features = features(data.index, mode, bins, db, queries, refit=refit)\n'
    '    precision, predicted, timings = [], [], []\n'
    '    labels = data.labels[db]\n'
    '    actual = data.labels[queries]\n'
    '    k = min(5, len(db))\n'
    '    for query, target in zip(query_features, actual):\n'
    '        start = data.clock()\n'
    '        order, distances = search_topk(query, database, k, metric)\n'
    '        timings.append(data.clock() - start)\n'
    '        neighbors = labels[order]\n'
    '        precision.append(float(np.mean(neighbors == target)))\n'
    '        predicted.append(knn_predict(neighbors, distances))\n'
    '    classes = np.asarray(classes)\n'
    '    actual_ids = np.searchsorted(classes, actual)\n'
    '    p_class = [float(np.mean(np.asarray(precision)[actual == cls])) for cls in classes]\n'
    '    names = [CLASSES[cls] for cls in classes]\n'
    '    payload = {\n'
    '        "mode": mode, "metric": metric, "bins": bins if mode in COUNT_KEYS else None,\n'
    '        "protocol": {"n_queries": len(queries), "db_size": len(db), "k": k,\n'
    '                     "queries_per_class": int(np.bincount(actual_ids).max())},\n'
    '        "classes": names, "per_class": [\n'
    '            {"cls": name, "n_queries": int(np.count_nonzero(actual_ids == i)), "precision_at_k": value}\n'
    '            for i, (name, value) in enumerate(zip(names, p_class))\n'
    '        ],\n'
    '        "macro_precision_at_k": float(np.mean(p_class)),\n'
    '        "knn_accuracy": float(np.mean(np.asarray(predicted) == actual)),\n'
    '        "confusion_matrix": confusion_matrix(actual_ids, np.searchsorted(classes, predicted), len(classes)).tolist(),\n'
    '        "source": "precomputed",\n'
    '    }\n'
    '    return payload, float(np.mean(timings[:50]))\n'
    '\n'
    '\n'
    'def combinations():\n'
    '    for mode, info in MODE_INFO.items():\n'
    '        for metric in info["metrics"]:\n'
    '            for bins in BINS if mode == "rgb" else (32,):\n'
    '                yield mode, metric, bins\n'
    '\n'
    '\n'
    'def table(columns, rows):\n'
    '    return {"columns": columns, "rows": rows}\n'
    '\n'
    '\n'
    'def heatmap(title, entry):\n'
    '    return {"kind": "heatmap", "title": title, "z": entry["confusion_matrix"],\n'
    '            "x_labels": entry["classes"], "y_labels": entry["classes"]}\n'
    '\n'
    '\n'
    'def result(title, description, data, tables, findings, *, charts=(), params=None, n_queries=None):\n'
    '    return {"title": title, "description": description, "params": params or {},\n'
    '            "dataset": {"n_images": len(data.rows), "n_queries": n_queries,\n'
    '                        "note": "Fixture sintetis" if data.smoke else "EuroSAT lokal"},\n'
    '            "tables": tables, "charts": list(charts), "findings": findings,\n'
    '            "analysis_md": " ".join(findings)[:1500]}\n'
)

embedded_sources['src/experiments/run_all.py'] = (
    '"""Build offline experiment artifacts for a synthetic smoke run or full EuroSAT."""\n'
    'import argparse\n'
    'import json\n'
    'from datetime import datetime, timezone\n'
    'from pathlib import Path\n'
    'from tempfile import TemporaryDirectory\n'
    'from time import perf_counter\n'
    '\n'
    'import numpy as np\n'
    'import tifffile\n'
    'from PIL import Image\n'
    '\n'
    'from src.core.cbir import MODE_INFO, zscore_fit\n'
    'from src.data.env import PROJECT_ROOT, load_settings\n'
    'from src.experiments import c1_bins, c2_metrics, c3_bitflip, c4a_modalities, c4b_partial_shares, c4c_complexity\n'
    'from src.experiments.common import Dataset, combinations, evaluate\n'
    'from tools.build_index import extract_features, fit_scale, read_manifest\n'
    '\n'
    'C4A_MODES = tuple(MODE_INFO)\n'
    '\n'
    '\n'
    'def _json_value(value):\n'
    '    if isinstance(value, np.generic):\n'
    '        value = value.item()\n'
    '    if isinstance(value, float):\n'
    '        return round(value, 6)\n'
    '    if isinstance(value, dict):\n'
    '        return {key: _json_value(item) for key, item in value.items()}\n'
    '    if isinstance(value, (tuple, list)):\n'
    '        return [_json_value(item) for item in value]\n'
    '    return value\n'
    '\n'
    '\n'
    'def run(data, destination, *, c4a_modes=C4A_MODES, c4b_mode="rgb", generated_at=None):\n'
    '    if generated_at is None:\n'
    '        generated_at = datetime.now(timezone.utc).isoformat()\n'
    '    cache = {}\n'
    '\n'
    '    def evaluation(mode, metric, bins):\n'
    '        key = f"{mode}:{metric}:{bins}"\n'
    '        if key not in cache:\n'
    '            cache[key] = evaluate(data, mode, metric, bins)\n'
    '        return cache[key]\n'
    '\n'
    '    jobs = (\n'
    '        ("C1", lambda: c1_bins.run(data, evaluation)),\n'
    '        ("C2", lambda: c2_metrics.run(data, evaluation)),\n'
    '        ("C3", lambda: c3_bitflip.run(data)),\n'
    '        ("C4a", lambda: c4a_modalities.run(data, c4a_modes)),\n'
    '        ("C4b", lambda: c4b_partial_shares.run(data, c4b_mode)),\n'
    '        ("C4c", lambda: c4c_complexity.run(data)),\n'
    '    )\n'
    '    payloads = {}\n'
    '    for name, execute in jobs:\n'
    '        start = data.clock()\n'
    '        payload = execute()\n'
    '        payload.update(id=name, generated_at=generated_at, runtime_s=data.clock() - start)\n'
    '        payload["params"].update(seed=data.seed, all_queries=data.all_queries)\n'
    '        payloads[name] = payload\n'
    '        print(f"{name}: {payload[\'runtime_s\']:.3f}s", flush=True)\n'
    '\n'
    '    for args in combinations():\n'
    '        evaluation(*args)\n'
    '    payloads["eval_cache"] = {key: entry for key, (entry, _) in cache.items()}\n'
    '    destination.mkdir(parents=True, exist_ok=True)\n'
    '    with TemporaryDirectory(dir=destination) as temp:\n'
    '        staging = Path(temp)\n'
    '        for name, payload in payloads.items():\n'
    '            file = f"{name}.json"\n'
    '            (staging / file).write_text(\n'
    '                json.dumps(_json_value(payload), ensure_ascii=False, allow_nan=False, indent=2) + "\\n",\n'
    '                encoding="utf-8",\n'
    '            )\n'
    '        for path in staging.iterdir():\n'
    '            path.replace(destination / path.name)\n'
    '\n'
    '\n'
    'def _make_smoke_data(root, seed, clock):\n'
    '    rng = np.random.default_rng(seed)\n'
    '    rows = []\n'
    '    labels, splits = [], []\n'
    '    for cls in (0, 1, 2, 5):\n'
    '        for i in range(10):\n'
    '            image_id = f"sample_{cls}_{i}"\n'
    '            rgb = rng.integers(cls * 20, cls * 20 + 100, (8, 8, 3), dtype=np.uint8)\n'
    '            bands = rng.integers(cls * 200, cls * 200 + 2000, (8, 8, 13), dtype=np.uint16)\n'
    '            jpg, tif = f"{image_id}.jpg", f"{image_id}.tif"\n'
    '            Image.fromarray(rgb).save(root / jpg)\n'
    '            tifffile.imwrite(root / tif, bands, photometric="minisblack")\n'
    '            test = int(i >= 8)\n'
    '            labels.append(cls)\n'
    '            splits.append(test)\n'
    '            rows.append(dict(id=image_id, rgb_path=jpg, tif_path=tif, split="test" if test else "train"))\n'
    '\n'
    '    scale = fit_scale(rows, root, seed)\n'
    '    all_features = {}\n'
    '    data = Dataset({}, rows, root, root, clock, seed=seed, smoke=True)\n'
    '    for i in range(len(rows)):\n'
    '        counts, moments = extract_features(data.rgb(i), data.bands(i), scale)\n'
    '        for prefix, values in (("counts", counts), ("feat", moments)):\n'
    '            for key, value in values.items():\n'
    '                all_features.setdefault(f"{prefix}_{key}", []).append(value)\n'
    '    index = {\n'
    '        name: np.stack(values).astype(np.uint16 if name.startswith("counts_") else np.float32)\n'
    '        for name, values in all_features.items()\n'
    '    }\n'
    '    index.update(ids=np.array([row["id"] for row in rows]), cls_idx=np.array(labels), split=np.array(splits))\n'
    '    for name in MODE_INFO:\n'
    '        if MODE_INFO[name]["kind"] == "moment":\n'
    '            mean, std = zscore_fit(index[f"feat_{name}"][index["split"] == 0])\n'
    '            index[f"z_{name}_mean"], index[f"z_{name}_std"] = mean.astype(np.float32), std.astype(np.float32)\n'
    '    data.index = index\n'
    '    return data\n'
    '\n'
    '\n'
    'def run_smoke(destination, *, c4a_modes=C4A_MODES, c4b_mode="rgb", clock=perf_counter, generated_at=None):\n'
    '    destination.parent.mkdir(parents=True, exist_ok=True)\n'
    '    with TemporaryDirectory(dir=destination.parent) as temp:\n'
    '        run(_make_smoke_data(Path(temp), 42, clock), destination, c4a_modes=c4a_modes,\n'
    '            c4b_mode=c4b_mode, generated_at=generated_at)\n'
    '\n'
    '\n'
    'def main():\n'
    '    parser = argparse.ArgumentParser(description=__doc__)\n'
    '    parser.add_argument("--config", choices=("smoke", "full"), required=True)\n'
    '    parser.add_argument("--all-queries", action="store_true")\n'
    '    parser.add_argument("--output", type=Path)\n'
    '    args = parser.parse_args()\n'
    '    if args.config == "smoke":\n'
    '        if args.all_queries:\n'
    '            parser.error("--all-queries hanya diperlukan untuk konfigurasi full")\n'
    '        run_smoke(args.output or PROJECT_ROOT / "results" / "smoke")\n'
    '        return\n'
    '\n'
    '    settings = load_settings()\n'
    '    if settings.rgb_dir is None or settings.allbands_dir is None:\n'
    '        parser.error("Konfigurasikan direktori RGB dan all-bands")\n'
    '    rows = read_manifest(settings.artifacts_dir / "manifest.csv")\n'
    '    with np.load(settings.artifacts_dir / "index.npz", allow_pickle=False) as saved:\n'
    '        index = {key: saved[key] for key in saved.files if key != "thumb_off"}\n'
    '    if len(rows) != 27000 or index["ids"].tolist() != [row["id"] for row in rows]:\n'
    '        parser.error("Konfigurasi full memerlukan indeks yang cocok dengan manifest 27.000 citra")\n'
    '    data = Dataset(index, rows, settings.rgb_dir, settings.allbands_dir, perf_counter,\n'
    '                   seed=settings.seed, all_queries=args.all_queries)\n'
    '    run(data, args.output or PROJECT_ROOT / "results")\n'
    '\n'
    '\n'
    'if __name__ == "__main__":\n'
    '    main()\n'
)

embedded_sources['tools/build_index.py'] = (
    '"""Build histograms, moment features and an original JPEG pack."""\n'
    'import argparse\n'
    'import csv\n'
    'import json\n'
    'import multiprocessing as mp\n'
    'import sys\n'
    'from pathlib import Path\n'
    'from tempfile import TemporaryDirectory\n'
    '\n'
    'if __package__ in (None, ""):\n'
    '    sys.path.insert(0, str(Path(__file__).resolve().parents[1]))\n'
    '\n'
    'import numpy as np\n'
    '\n'
    'from src.core.cbir import extract_counts, extract_moment_feature, zscore_fit\n'
    'from src.core.spectral import fit_percentile_scale\n'
    'from src.data.env import load_settings\n'
    'from src.data.loader import load_rgb, load_tif\n'
    '\n'
    'COUNT_SHAPES = {"gray": (), "rgb": (3,), "hsv": (3,), "ms": (13,), "ndvi": (), "ndwi": ()}\n'
    'MOMENT_DIMS = {"grid2": 36, "grid4": 144, "mle": 6, "ms_mle": 26}\n'
    '_worker_inputs = None\n'
    '\n'
    '\n'
    'def read_manifest(path):\n'
    '    with path.open(newline="", encoding="utf-8") as stream:\n'
    '        return list(csv.DictReader(stream))\n'
    '\n'
    '\n'
    'def fit_scale(rows, tif_root, seed):\n'
    '    train = [row["tif_path"] for row in rows if row["split"] == "train"]\n'
    '    rng = np.random.default_rng(seed)\n'
    '    selected = rng.choice(len(train), size=min(2000, len(train)), replace=False)\n'
    '    return fit_percentile_scale([load_tif(tif_root / train[i]) for i in selected])\n'
    '\n'
    '\n'
    'def _init_worker(rgb_root, tif_root, scale):\n'
    '    global _worker_inputs\n'
    '    _worker_inputs = rgb_root, tif_root, scale\n'
    '\n'
    '\n'
    'def extract_features(rgb, bands, scale):\n'
    '    kwargs = dict(rgb=rgb, bands=bands, scale=scale)\n'
    '    counts = {name: extract_counts(name, **kwargs) for name in ("gray", "rgb", "hsv")}\n'
    '    counts["ms"] = extract_counts("ms13", **kwargs)\n'
    '    counts["ndvi"], counts["ndwi"] = extract_counts("ndvi_ndwi", **kwargs)\n'
    '    return counts, {name: extract_moment_feature(name, **kwargs) for name in MOMENT_DIMS}\n'
    '\n'
    '\n'
    'def _extract(row):\n'
    '    rgb_root, tif_root, scale = _worker_inputs\n'
    '    path = rgb_root / row["rgb_path"]\n'
    '    rgb, bands = load_rgb(path), load_tif(tif_root / row["tif_path"])\n'
    '    if rgb.shape[:2] != bands.shape[:2] or rgb.shape[0] * rgb.shape[1] > 65535:\n'
    '        raise ValueError(f"Incompatible image sizes for {row[\'id\']}")\n'
    '    counts, moments = extract_features(rgb, bands, scale)\n'
    '    return counts, moments, path.read_bytes()\n'
    '\n'
    '\n'
    'def build_index(manifest, rgb_root, tif_root, destination, *, seed=42):\n'
    '    rows = read_manifest(manifest)\n'
    '    ids = [row["id"] for row in rows]\n'
    '    if not rows or len(set(ids)) != len(ids) or any(len(key) > 32 for key in ids):\n'
    '        raise ValueError("Expected unique image IDs of at most 32 characters")\n'
    '    if any(not row["rgb_path"] or not row["tif_path"] for row in rows):\n'
    '        raise ValueError("The index requires paired JPEG and TIFF images")\n'
    '\n'
    '    scale = fit_scale(rows, tif_root, seed)\n'
    '    split = np.array([{"train": 0, "test": 1}[row["split"]] for row in rows], dtype=np.int8)\n'
    '    n = len(rows)\n'
    '    offsets = np.zeros(n + 1, dtype=np.int64)\n'
    '    payload = dict(ids=np.array(ids, dtype="U32"), split=split,\n'
    '                   cls_idx=np.array([int(row["cls_idx"]) for row in rows], dtype=np.int8))\n'
    '    payload["thumb_off"] = offsets\n'
    '    for name, shape in COUNT_SHAPES.items():\n'
    '        payload[f"counts_{name}"] = np.empty((n, *shape, 256), dtype=np.uint16)\n'
    '    for name, dim in MOMENT_DIMS.items():\n'
    '        payload[f"feat_{name}"] = np.empty((n, dim), dtype=np.float32)\n'
    '\n'
    '    destination.mkdir(parents=True, exist_ok=True)\n'
    '    with TemporaryDirectory(dir=destination) as folder:\n'
    '        staging = Path(folder)\n'
    '        with (staging / "thumbs.pack").open("wb") as pack, mp.get_context("spawn").Pool(\n'
    '            4, initializer=_init_worker, initargs=(rgb_root, tif_root, scale),\n'
    '        ) as pool:\n'
    '            for i, (counts, moments, jpeg) in enumerate(pool.imap(_extract, rows, chunksize=16)):\n'
    '                for name, value in counts.items():\n'
    '                    payload[f"counts_{name}"][i] = value\n'
    '                for name, value in moments.items():\n'
    '                    payload[f"feat_{name}"][i] = value\n'
    '                pack.write(jpeg)\n'
    '                offsets[i + 1] = pack.tell()\n'
    '        for name in MOMENT_DIMS:\n'
    '            stats = zscore_fit(payload[f"feat_{name}"][split == 0])\n'
    '            for field, value in zip(("mean", "std"), stats):\n'
    '                payload[f"z_{name}_{field}"] = value.astype(np.float32)\n'
    '        np.savez_compressed(staging / "index.npz", **payload)\n'
    '        (staging / "scale.json").write_text(json.dumps(scale) + "\\n", encoding="utf-8")\n'
    '        for name in ("scale.json", "thumbs.pack", "index.npz"):\n'
    '            (staging / name).replace(destination / name)\n'
    '\n'
    '\n'
    'def main():\n'
    '    parser = argparse.ArgumentParser(description=__doc__)\n'
    '    parser.parse_args()\n'
    '    settings = load_settings()\n'
    '    if settings.rgb_dir is None or settings.allbands_dir is None:\n'
    '        parser.error("Konfigurasikan direktori RGB dan all-bands")\n'
    '    build_index(settings.artifacts_dir / "manifest.csv", settings.rgb_dir, settings.allbands_dir,\n'
    '                settings.artifacts_dir, seed=settings.seed)\n'
    '\n'
    '\n'
    'if __name__ == "__main__":\n'
    '    main()\n'
)

embedded_sources['tools/colab_runtime.py'] = (
    '"""Data preparation and reporting used by the generated Colab notebook."""\n'
    'import hashlib\n'
    'import os\n'
    'import re\n'
    'import shutil\n'
    'from pathlib import Path, PurePosixPath\n'
    'from tempfile import TemporaryDirectory\n'
    'from urllib.request import urlopen\n'
    'from zipfile import ZipFile\n'
    '\n'
    'from dotenv import load_dotenv\n'
    'import numpy as np\n'
    '\n'
    'from src.experiments.common import Dataset\n'
    'from tools.build_index import read_manifest\n'
    'from time import perf_counter\n'
    '\n'
    'ARCHIVE_NAME = "eurosat_subset_v1.zip"\n'
    'EXPERIMENTS = ("C1", "C2", "C3", "C4a", "C4b", "C4c")\n'
    '\n'
    '\n'
    'def _sha256(path):\n'
    '    with path.open("rb") as file:\n'
    '        return hashlib.file_digest(file, "sha256").hexdigest()\n'
    '\n'
    '\n'
    'def download_subset(cache, urls, digest=""):\n'
    '    """Fetch a release (primary then mirror) and verify its published digest."""\n'
    '    cache.mkdir(parents=True, exist_ok=True)\n'
    '    destination = cache / ARCHIVE_NAME\n'
    '    errors = []\n'
    '    for url in filter(None, urls):\n'
    '        try:\n'
    '            expected = digest\n'
    '            if not expected:\n'
    '                with urlopen(url + ".sha256", timeout=60) as stream:\n'
    '                    fields = stream.read(4096).decode("ascii").split()\n'
    '                expected = fields[0] if fields else ""\n'
    '            if not re.fullmatch("[0-9a-fA-F]{64}", expected):\n'
    '                raise ValueError("SHA-256 rilis tidak valid")\n'
    '            expected = expected.lower()\n'
    '            if destination.is_file() and _sha256(destination) == expected:\n'
    '                return destination\n'
    '            with TemporaryDirectory(dir=cache) as folder:\n'
    '                stage = Path(folder) / ARCHIVE_NAME\n'
    '                with urlopen(url, timeout=60) as stream, stage.open("wb") as target:\n'
    '                    shutil.copyfileobj(stream, target)\n'
    '                if _sha256(stage) != expected:\n'
    '                    raise ValueError("SHA-256 arsip berbeda dari rilis")\n'
    '                stage.replace(destination)\n'
    '            return destination\n'
    '        except (OSError, ValueError) as exc:\n'
    '            errors.append(str(exc))\n'
    '    raise RuntimeError("Arsip tidak tersedia. Atur COLAB_ARCHIVE atau COLAB_SUBSET_URL. " + " | ".join(errors))\n'
    '\n'
    '\n'
    'def extract_verified(archive, cache, digest=""):\n'
    '    """Read checked members into a cache keyed by the verified ZIP contents."""\n'
    '    archive_digest = _sha256(archive)\n'
    '    sidecar = archive.with_name(archive.name + ".sha256")\n'
    '    fields = sidecar.read_text(encoding="ascii").split() if sidecar.is_file() else []\n'
    '    expected = digest or (fields[0] if fields else archive_digest)\n'
    '    if expected.lower() != archive_digest:\n'
    '        raise ValueError("Checksum ZIP tidak cocok")\n'
    '\n'
    '    destination = cache / archive_digest\n'
    '    with ZipFile(archive) as bundle:\n'
    '        entries = bundle.namelist()\n'
    '        checks = {}\n'
    '        for line in bundle.read("SHA256SUMS").decode("utf-8").splitlines():\n'
    '            digest, name = line.split("  ", 1)\n'
    '            path = PurePosixPath(name)\n'
    '            if (not re.fullmatch("[0-9a-f]{64}", digest) or not path.parts or path.is_absolute()\n'
    '                    or ".." in path.parts or ":" in name or "\\\\" in name or name in checks):\n'
    '                raise ValueError(f"Entri checksum tidak valid: {name}")\n'
    '            checks[name] = digest\n'
    '        if len(entries) != len(set(entries)) or set(entries) != set(checks) | {"SHA256SUMS"}:\n'
    '            raise ValueError("Daftar anggota ZIP berbeda dari SHA256SUMS")\n'
    '        if destination.is_dir():\n'
    '            for name, digest in checks.items():\n'
    '                if _sha256(destination / name) != digest:\n'
    '                    raise ValueError(f"Checksum cache tidak cocok: {name}")\n'
    '            return destination\n'
    '\n'
    '        cache.mkdir(parents=True, exist_ok=True)\n'
    '        with TemporaryDirectory(dir=cache) as folder:\n'
    '            stage = Path(folder) / "dataset"\n'
    '            for name, digest in checks.items():\n'
    '                path = stage / name\n'
    '                path.parent.mkdir(parents=True, exist_ok=True)\n'
    '                with bundle.open(name) as stream, path.open("wb") as target:\n'
    '                    shutil.copyfileobj(stream, target)\n'
    '                if _sha256(path) != digest:\n'
    '                    raise ValueError(f"Checksum anggota tidak cocok: {name}")\n'
    '            stage.replace(destination)\n'
    '    return destination\n'
    '\n'
    '\n'
    'def dataset_from_mode(mode, workspace, cache, *, archive=None, urls=(), digest=""):\n'
    '    if mode == "colab_instant":\n'
    '        bundle = archive if archive is not None else download_subset(cache, urls, digest)\n'
    '        root = extract_verified(bundle, cache / "datasets", digest)\n'
    '        rgb, tif, artifacts = root / "rgb", root / "tif", root\n'
    '    elif mode == "full_local":\n'
    '        load_dotenv(workspace / ".env", override=False)\n'
    '        rgb_dir, tif_dir = os.getenv("EUROSAT_RGB_DIR"), os.getenv("EUROSAT_ALLBANDS_DIR")\n'
    '        if not rgb_dir or not tif_dir:\n'
    '            raise ValueError("Isi EUROSAT_RGB_DIR dan EUROSAT_ALLBANDS_DIR untuk full_local.")\n'
    '        rgb, tif = workspace / Path(rgb_dir).expanduser(), workspace / Path(tif_dir).expanduser()\n'
    '        artifacts = workspace / (os.getenv("ARTIFACTS_DIR") or "artifacts")\n'
    '    else:\n'
    '        raise ValueError(f"Mode dataset tidak dikenal: {mode}")\n'
    '\n'
    '    rows = read_manifest(artifacts / "manifest.csv")\n'
    '    with np.load(artifacts / "index.npz", allow_pickle=False) as saved:\n'
    '        index = {key: saved[key] for key in saved.files if key != "thumb_off"}\n'
    '    if index["ids"].tolist() != [row["id"] for row in rows]:\n'
    '        raise ValueError("ID indeks dan manifest berbeda.")\n'
    '    return Dataset(index, rows, rgb, tif, perf_counter, seed=42, all_queries=mode == "colab_instant"), artifacts\n'
)

In [ ]:
source_tree = TemporaryDirectory(prefix="citrasatelit-")
project = Path(source_tree.name) / "project"
if REPO_URL:
    subprocess.run(["git", "clone", "--depth", "1", "--", REPO_URL, str(project)], check=True)
else:
    for relative, source in embedded_sources.items():
        file = project / relative
        file.parent.mkdir(parents=True, exist_ok=True)
        file.write_text(source, encoding="utf-8")
sys.path.insert(0, str(project))

import json

import matplotlib.pyplot as plt
import numpy as np

from src.core import boolean_lab, cbir, color, probability, secret_sharing, security_demos, spectral, stats
from src.experiments import run_all
from src.experiments.common import evaluate
from tools.colab_runtime import ARCHIVE_NAME, EXPERIMENTS, dataset_from_mode

## 3. Data dan manifest
Checksum anggota diverifikasi sebelum data dipakai. ID indeks diperiksa terhadap manifest.

In [ ]:
archive_candidates = [Path(COLAB_ARCHIVE)] if COLAB_ARCHIVE else [
    workspace / "artifacts" / ARCHIVE_NAME, workspace / ARCHIVE_NAME, Path("/content") / ARCHIVE_NAME,
]
if DATASET_MODE == "colab_instant" and COLAB_ARCHIVE and not archive_candidates[0].is_file():
    raise FileNotFoundError(f"COLAB_ARCHIVE tidak ditemukan: {COLAB_ARCHIVE}")
archive = next((file for file in archive_candidates if file.is_file()), None)
data, artifacts = dataset_from_mode(DATASET_MODE, workspace, cache, archive=archive,
                                    urls=(COLAB_SUBSET_URL, COLAB_SUBSET_MIRROR), digest=COLAB_SUBSET_SHA256)
train = data.split_rows(0, data.classes)
test = data.split_rows(1, data.classes)
if DATASET_MODE == "colab_instant":
    assert len(train) == 1600 and len(test) == 400
else:
    assert len(data.rows) == 27000
print(f"{DATASET_MODE} · {len(train):,} latih · {len(test):,} uji")
i = test[0]
rgb, bands = data.rgb(i), data.bands(i)
scale = json.loads((artifacts / "scale.json").read_text(encoding="utf-8"))

def show_figure(fig):
    fig.tight_layout()
    fig.canvas.draw()
    if plt.get_backend().lower() != "agg":
        plt.show()
    plt.close(fig)

## 4. Histogram, momen dan spektrum
Histogram yang dinormalisasi adalah PMF. NDVI dan NDWI memakai kanal berbeda.

In [ ]:
gray = color.rgb_to_gray(rgb)
counts = stats.histogram_counts(gray, 256)
pmf = stats.counts_to_pmf(counts)
moments = stats.moments_from_counts(counts)
assert np.isclose(pmf.sum(), 1) and np.isclose(moments["mean"], gray.mean())
assert np.isclose(stats.mle_gaussian(gray)[0], gray.mean())
mgf = stats.mgf_check(counts)
assert np.isclose(mgf["m1_fd"], mgf["m1_exact"], atol=1e-5)
assert color.rgb_to_hsv(rgb).shape == rgb.shape
assert spectral.synth_rgb(bands, scale).shape == rgb.shape
fig, axes = plt.subplots(1, 3, figsize=(11, 3))
axes[0].bar(np.arange(256), pmf)
axes[0].set_title("PMF keabuan")
for axis, image, title in zip(axes[1:], (spectral.ndvi(bands), spectral.ndwi(bands)), ("NDVI", "NDWI")):
    shown = axis.imshow(image, vmin=-1, vmax=1)
    axis.set_title(title)
    axis.axis("off")
    fig.colorbar(shown, ax=axis)
show_figure(fig)
print(moments)

## 5. CBIR dan Precision@5
Kueri berasal dari split uji. Tetangga dicari hanya di split latih.

In [ ]:
summary, _ = evaluate(data, "rgb", "chi2", 32)
assert 0 <= summary["macro_precision_at_k"] <= 1
assert cbir.extract_counts("rgb", rgb=rgb).sum() == rgb.size
for row in summary["per_class"]:
    print(f'{row["cls"]:<22} {row["precision_at_k"]:.1%} ({row["n_queries"]} kueri)')
print(f'P@5 makro {summary["macro_precision_at_k"]:.1%}')

## 6. Teorema Bayes
Bandingkan peluang awal dengan posterior kasus Transjakarta dan Gateway.

In [ ]:
posterior, evidence = probability.bayes_posterior([0.60, 0.40], [0.10, 0.05])
assert np.isclose(evidence, 0.08) and np.isclose(posterior[0], 0.75)
assert np.isclose(probability.cond_prob(0.06, 0.08), 0.75)
gateway, probability_error = probability.bayes_posterior([0.70, 0.30], [0.02, 0.08])
assert np.isclose(probability_error, 0.038) and np.isclose(gateway[1], 0.024 / 0.038)
print("P(T) =", evidence, "· P(TL | T) =", posterior[0], "· P(Y | E) =", gateway[1])

## 7. Tiga share XOR
Gabungkan semua share untuk memulihkan citra. Paritas XOR juga dapat memulihkan satu blok yang hilang.

In [ ]:
shares = secret_sharing.split_xor(bands[..., 3], 3, np.random.default_rng(data.seed))
restored = secret_sharing.reconstruct_xor(shares)
assert np.array_equal(restored, bands[..., 3]) and secret_sharing.mse(restored, bands[..., 3]) == 0
assert np.isinf(secret_sharing.psnr(restored, bands[..., 3], 65535))
parity = security_demos.raid5_parity(shares)
assert np.array_equal(security_demos.raid5_recover([shares[0], None, shares[2]], parity, 1), shares[1])
fig, axes = plt.subplots(1, 4, figsize=(10, 3))
for axis, image, title in zip(axes, [*shares, restored], ["Share 1", "Share 2", "Share 3", "Pulih"]):
    axis.imshow(image, cmap="gray")
    axis.set_title(title)
    axis.axis("off")
show_figure(fig)

## 8. Aljabar Boolean dan peta Karnaugh
Uji identitas XOR, De Morgan, dan fungsi mayoritas.

In [ ]:
assert boolean_lab.equivalent("ab'+a'b", "a^b")
assert boolean_lab.equivalent("(a+b)'", "a'b'")
assert boolean_lab.equivalent(boolean_lab.dual_expr("a+a'b"), boolean_lab.dual_expr("a+b"))
truth = boolean_lab.truth_table("a^b")
assert [row["output"] for row in truth["rows"]] == [0, 1, 1, 0]
majority = boolean_lab.kmap(3, [3, 5, 6, 7])
assert boolean_lab.equivalent(majority["simplified"], "ab+ac+bc")
print("Tabel XOR:", truth["rows"], "· SOP mayoritas:", majority["simplified"])

## 9. Percobaan C1–C4c
Hitung ulang enam percobaan dengan data aktif. Keluaran JSON berisi tabel lengkap dan parameter.
Pratinjau menampilkan hingga 12 baris setiap tabel. Pada C4b, grafik dibuat dari kolom P@5 di tabel hasil.

In [ ]:
output = cache / "results" / DATASET_MODE
run_all.run(data, output)
results = {name: json.loads((output / (name + ".json")).read_text(encoding="utf-8")) for name in EXPERIMENTS}
assert all(item["tables"] and item["findings"] for item in results.values())

In [ ]:
def show_text(text):
    if "ipykernel" in sys.modules:
        from IPython.display import Markdown, display
        display(Markdown(text))
    else:
        print(text)

def show_chart(chart):
    fig, axis = plt.subplots(figsize=(7, 3.8))
    axis.set_title(chart["title"])
    if chart["kind"] == "heatmap":
        image = axis.imshow(chart["z"], cmap="Blues")
        xlabels, ylabels = chart["x_labels"], chart["y_labels"]
        axis.set_xticks(range(len(xlabels)), xlabels, rotation=55, ha="right")
        axis.set_yticks(range(len(ylabels)), ylabels)
        fig.colorbar(image, ax=axis)
    else:
        for series in chart["series"]:
            x = series.get("x", chart.get("x", list(range(len(series["y"])))))
            if chart["kind"] == "bar":
                axis.bar(x, series["y"], label=series["name"])
            elif chart["kind"] == "scatter":
                axis.scatter(x, series["y"], label=series["name"])
            else:
                axis.plot(x, series["y"], marker="o", label=series["name"])
        if len(chart["series"]) > 1:
            axis.legend()
    show_figure(fig)

for identifier, item in results.items():
    show_text("### " + item["title"])
    for table in item["tables"]:
        columns, rows = table["columns"], table["rows"]
        lines = ["| " + " | ".join(columns) + " |", "| " + " | ".join(["---"] * len(columns)) + " |"]
        lines.extend("| " + " | ".join(map(str, row)) + " |" for row in rows[:12])
        show_text(chr(10).join(lines))
        if len(rows) > 12:
            show_text(f"Baris lengkap ({len(rows)}) tersedia di {identifier}.json.")
    for chart in item["charts"]:
        show_chart(chart)
    if identifier == "C4b":
        rows = item["tables"][0]["rows"]
        series = []
        for n in sorted({row[0] for row in rows}):
            selected = [row for row in rows if row[0] == n and row[2] == "awal"]
            series.append({"name": f"n={n}", "x": [row[1] for row in selected], "y": [row[7] for row in selected]})
        show_chart({"kind": "line", "title": "Precision@5 menurut jumlah share yang digabung", "series": series})

## 10. Analisis
Temuan dihitung dari keluaran setiap percobaan.

In [ ]:
for identifier, item in results.items():
    show_text(f"### {identifier}" + chr(10) + chr(10).join("- " + finding for finding in item["findings"]))
print("Hasil lengkap:", output)

## 11. Kesimpulan
Bandingkan metode pada split yang sama sebelum menarik kesimpulan tentang kemiripan.
Semua share diperlukan untuk rekonstruksi identik. Peta Karnaugh menunjukkan bagaimana fungsi Boolean disederhanakan.

## 12. Transparansi AI
Pembangunan notebook, verifikasi, dan hasil pengukuran dicatat di **AI_USAGE_LOG.md** pada repositori proyek.